Imports

In [2]:
import oligoformer as ol
import os
from dotenv import load_dotenv
from Bio import Entrez

load_dotenv()

NCBI_EMAIL = os.getenv("NCBI_EMAIL")
if not NCBI_EMAIL:
    raise EnvironmentError(
        "A variável de ambiente NCBI_EMAIL não está definida. A API do NCBI Entrez exige um "
        "e-mail de contato válido para uso. Veja o README para instruções de configuração."
    )
Entrez.email = NCBI_EMAIL
Entrez.api_key = os.getenv("NCBI_API_KEY")
Entrez.sleep_between_tries = 1
Entrez.max_tries = 4
import dsir
import pathlib as ph
import re
import time
import pandas as pd
from typing import Literal


Definições das funções de API

Symbol -> Gene_id -> Uid -> Accession -> Fasta

In [3]:
def gene_symbol_to_gene_id(symbol: str, organism="Homo sapiens"):
    query = f"{symbol}[sym] AND {organism}[orgn]"
    
    handle = Entrez.esearch(db="gene", term=query)
    try:
        record = Entrez.read(handle)
    except Exception as e:
        raise RuntimeError(f"Entrez failed: {e}")
    handle.close()
    
    if not record["IdList"]:
        raise ValueError(f"No gene found for symbol {symbol}")
    
    return record["IdList"][0]
#   |
#   |
#   V
def gene_id_to_uid(gene_id: str):
    handle = Entrez.elink(
        dbfrom="gene",
        db="nuccore",
        id=gene_id,
        linkname="gene_nuccore_refseqrna"  # IMPORTANT: limits to RefSeq RNA
    )
    try:
        record = Entrez.read(handle)
    except Exception as e:
        raise RuntimeError(f"Entrez failed: {e}")
    handle.close()
    
    links = record[0]["LinkSetDb"]
    
    if not links:
        raise ValueError("No linked nucleotide records found")
    
    accessions = [link["Id"] for link in links[0]["Link"]]
    return accessions
#   |
#   |
#   V
def uids_to_accessions(uids):
    handle = Entrez.efetch(
        db="nuccore",
        id=",".join(uids),
        rettype="acc",
        retmode="text"
    )
    accessions = handle.read().strip().split("\n")
    handle.close()
    
    return accessions
#   |
#   |
#   V
def get_fasta_from_accession(accession: str, alias="", absolute_alias=False):
    if(absolute_alias and alias == ""):
        raise ValueError ("absolute alias usage requires an alias")
    if(absolute_alias):
        if(alias.endswith(".fasta")):
            output_file = alias
        else:
            output_file = alias + ".fasta"
    else:
        if(alias != ""):
            alias = alias + "_"
        output_file = alias + accession + ".fasta"
    """
    Get the nucleotide FASTA for a given RefSeq accession, it contains the transcript sequence
    allways ends in .fasta
    """
    if(alias.endswith(".fasta")):
        alias = ".".join(alias.split('.')[:-1])
    alias = re.sub(r'[^a-zA-Z0-9_\-]', '_', alias) #sanitização final
    handle = Entrez.efetch(db="nuccore", id=accession, rettype="fasta", retmode="text")
    seq = handle.read()
    handle.close()
    #print(seq.split('\n'))
    #Sanitização extrema do header do arquivo fasta
    seq = seq.split('\n') #separa as linhas 
    seq[0] = f">{alias}_{seq[0].split()[0].lstrip('>')}" #pega o header e tira tudo menos a primeira palavra e depois adiciona o alias
    seq = '\n'.join(seq)
    with open(output_file, "w") as f:
        f.write(seq)
    return output_file
#   |
#   X (multiple accessions)
#   V
def get_multifasta_from_accessions(accessions: list, alias="", absolute_alias=False):
    if absolute_alias and alias == "":
        raise ValueError("absolute alias usage requires an alias")

    if absolute_alias:
        output_file = alias if alias.endswith(".fasta") else alias + ".fasta"
    else:
        if alias != "":
            alias = alias + "_"
        output_file = alias + accessions[0] + ".fasta"
    alias = re.sub(r'[^a-zA-Z0-9_\-]', '_', alias) #sanitização final
    handle = Entrez.efetch(
        db="nuccore",
        id=",".join(accessions),
        rettype="fasta",
        retmode="text"
    )
    data = handle.read()
    handle.close()

    # Optional: sanitize headers like you were doing
    lines = data.split("\n")
    for i in range(len(lines)):
        if lines[i].startswith(">"):
            acc = lines[i].split()[0].lstrip(">")
            lines[i] = f">{alias}_{acc}"

    data = "\n".join(lines)

    with open(output_file, "w") as f:
        f.write(data)

    return output_file

In [4]:
def choice(l: list):
    print(l)
    while(True):
        i = int(input())
        if(i >= 0 and i < len(l)):
            print(l[i], "chosen")
            return l[i]
        else:
            print("Please choose a valid number")


def accession_choice(acc: list, choose=False, XM_allow=False):
    l = []
    if len(acc) == 0:
        raise ValueError ("No transcripts found")
    if(choose == False):
        print("Transcript choice is disabled, using first available transcript according to configuration,\n please note this will default to NM only if possible")
    if(XM_allow):
        if(choose):
            print(f"XM transcripts allowed, showing all available transcripts recovered for the gene, chosse one from 0 to {len(acc)-1}")
            return choice(acc)
        else:
            print("Using transcript:", acc[0])
            return acc[0]
    else:
        for x in acc:
            if(x.startswith("NM_")):
                l.append(x)

        if (len(l) == 0): #No NM 
            if(choose):
                print(f"No NM seqs found for this gene, please choose one XM transcript from 0 to {len(l)-1}")
                return choice(l)
            else:
                for x in acc:
                    if(x.startswith("XM_")):
                        print("Using:", x, "no NM transcripts found")
                        return x
                raise ValueError("No NM or XM transcrips found, only other categories are available")
        elif(len(l) == 1): #one NM
            if(choose):
                print("Only one NM transcript found, defaulting to it. (To use XMs, XM_allow=True)")
            print("Using:", l[0])
            return l[0]
        else: #more than one NM
            if(choose):
                print(f"Multiple NM sequences found, please choose one from 0 to {len(l)-1}:")
                return choice(l)
            else:
                print("Using:", l[0])
                return l[0]
            

def symbol_to_fasta(sym: str, choose=False, XM_allow=False, alias="", absolute_alias=False, all_transcripts_multifasta=False):
    if(all_transcripts_multifasta):
        if(absolute_alias and alias == ""):
            raise ValueError ("absolute alias usage requires an alias")
        if(alias == ""):
            print("Warning: for fasta files with multiple transcripts, the use of aliases is highly recommended (use alias = *name* and absolute_alias = True)")
        accs = uids_to_accessions(gene_id_to_uid(gene_symbol_to_gene_id(sym)))
        if(XM_allow == False):
            l = []
            for acc in accs:
                if(acc.startswith("NM_")):
                    l.append(acc)
            accs = l
        if(alias != ""): #if no alias provided, use symbol as alias, not absolute
            sym = alias
        return get_multifasta_from_accessions(accs, alias=sym, absolute_alias=absolute_alias)    
    else:    
        if(absolute_alias and alias == ""):
            raise ValueError ("absolute alias usage requires an alias")
        acc = accession_choice(uids_to_accessions(gene_id_to_uid(gene_symbol_to_gene_id(sym))), choose=choose, XM_allow=XM_allow)
        if(alias != ""): #se alias é nada, usa o símbolo do gene é usado para diferenciar os arquivos
            sym = alias
        return get_fasta_from_accession(acc, alias=sym, absolute_alias=absolute_alias)

def list_transcripts(sym: str):
    acc = uids_to_accessions(gene_id_to_uid(gene_symbol_to_gene_id(sym)))
    print(acc)
    return acc


# TESTES

In [ ]:
#Teste de conexão básico
print("Email:", Entrez.email)
print("API key set:", Entrez.api_key is not None)

# simple query de teste
handle = Entrez.esearch(db="gene", term="BRCA1[gene] AND Homo sapiens[orgn]")
record = Entrez.read(handle)
handle.close()

assert "IdList" in record, "Invalid Entrez response"
assert len(record["IdList"]) > 0, "No results returned"
time.sleep(0.1)
print("✅ Entrez basic connectivity OK")

In [6]:
# Known stable gene
TEST_SYMBOL = "BRCA1"

gene_id = gene_symbol_to_gene_id(TEST_SYMBOL)
assert isinstance(gene_id, str) and gene_id.isdigit()

uids = gene_id_to_uid(gene_id)
assert isinstance(uids, list) and len(uids) > 0

accessions = uids_to_accessions(uids)
assert isinstance(accessions, list) and len(accessions) > 0

print("Gene ID:", gene_id)
print("UID count:", len(uids))
print("Accessions (first 3):", accessions[:3])

print("✅ Pipeline OK")

Gene ID: 672
UID count: 368
Accessions (first 3): ['NM_001408467.1', 'NM_001408497.1', 'NM_001408505.1']
✅ Pipeline OK


In [7]:
import tempfile

TEST_ACC = "NM_007294"  # BRCA1 RefSeq (stable)

with tempfile.TemporaryDirectory() as tmpdir:
    cwd = os.getcwd()
    os.chdir(tmpdir)  # isolate file creation

    try:
        fname = get_fasta_from_accession(TEST_ACC, alias="test", absolute_alias=False)
        
        # verify file exists and looks like FASTA
        assert os.path.exists(fname), "FASTA file not created"
        
        with open(fname) as f:
            content = f.read()
        
        assert content.startswith(">"), "Invalid FASTA format"
        assert len(content) > 100, "FASTA content too small"

        print("FASTA file created:", fname)
        print("✅ FASTA fetch OK")

    finally:
        os.chdir(cwd)  # restore working dir

FASTA file created: test_NM_007294.fasta
✅ FASTA fetch OK


In [8]:
TEST_SYMBOL = "TP53"

with tempfile.TemporaryDirectory() as tmpdir:
    cwd = os.getcwd()
    os.chdir(tmpdir)

    try:
        fname = symbol_to_fasta(
            TEST_SYMBOL,
            choose=False,
            XM_allow=False,
            alias="tp53_test",
            absolute_alias=False,
            all_transcripts_multifasta=True
        )

        assert os.path.exists(fname)

        with open(fname) as f:
            content = f.read()

        assert content.count(">") >= 1, "No sequences in multifasta"

        print("Sequences found:", content.count(">"))
        print("✅ Multifasta OK")

    finally:
        os.chdir(cwd)

Sequences found: 25
✅ Multifasta OK


# Pipeline

In [13]:
symbol = "MAPT"
alias = "MAPT_transcripts.fasta"
fasta = "MAPT_transcripts.fasta"

In [14]:
list_transcripts(symbol)

['XM_054316146.1', 'XM_054316145.1', 'XM_054316144.1', 'XM_054316143.1', 'XM_054316142.1', 'XM_054316141.1', 'XM_054316140.1', 'XM_054316139.1', 'XM_054316138.1', 'XM_054316137.1', 'XM_054316136.1', 'XM_054316135.1', 'XM_054316134.1', 'XM_054316133.1', 'XM_054316132.1', 'XM_054316131.1', 'XM_054330118.1', 'XM_054330117.1', 'XM_054330116.1', 'XM_054330115.1', 'XM_054330114.1', 'XM_054330113.1', 'XM_054330112.1', 'XM_054330111.1', 'XM_054330110.1', 'XM_054330109.1', 'XM_054330108.1', 'XM_054330107.1', 'XM_054330106.1', 'XM_054330105.1', 'XM_054328572.1', 'XM_054328571.1', 'XM_054328570.1', 'XM_054328569.1', 'XM_054328568.1', 'XM_054328567.1', 'XM_054328566.1', 'XM_054328565.1', 'XM_054328564.1', 'XM_047436081.1', 'XM_047436080.1', 'XM_005257371.5', 'XM_047436079.1', 'XM_005257370.5', 'XM_005257369.5', 'XM_047436078.1', 'XM_047436077.1', 'XM_005257368.5', 'XM_047436076.1', 'XM_047436075.1', 'XM_005257367.5', 'XM_047436074.1', 'XM_005257366.4', 'XM_005257365.5', 'XM_005257362.5', 'NM_00120

['XM_054316146.1',
 'XM_054316145.1',
 'XM_054316144.1',
 'XM_054316143.1',
 'XM_054316142.1',
 'XM_054316141.1',
 'XM_054316140.1',
 'XM_054316139.1',
 'XM_054316138.1',
 'XM_054316137.1',
 'XM_054316136.1',
 'XM_054316135.1',
 'XM_054316134.1',
 'XM_054316133.1',
 'XM_054316132.1',
 'XM_054316131.1',
 'XM_054330118.1',
 'XM_054330117.1',
 'XM_054330116.1',
 'XM_054330115.1',
 'XM_054330114.1',
 'XM_054330113.1',
 'XM_054330112.1',
 'XM_054330111.1',
 'XM_054330110.1',
 'XM_054330109.1',
 'XM_054330108.1',
 'XM_054330107.1',
 'XM_054330106.1',
 'XM_054330105.1',
 'XM_054328572.1',
 'XM_054328571.1',
 'XM_054328570.1',
 'XM_054328569.1',
 'XM_054328568.1',
 'XM_054328567.1',
 'XM_054328566.1',
 'XM_054328565.1',
 'XM_054328564.1',
 'XM_047436081.1',
 'XM_047436080.1',
 'XM_005257371.5',
 'XM_047436079.1',
 'XM_005257370.5',
 'XM_005257369.5',
 'XM_047436078.1',
 'XM_047436077.1',
 'XM_005257368.5',
 'XM_047436076.1',
 'XM_047436075.1',
 'XM_005257367.5',
 'XM_047436074.1',
 'XM_0052573

In [15]:
fasta = symbol_to_fasta(sym=symbol, XM_allow=True, absolute_alias=True, all_transcripts_multifasta=True, alias=alias)

In [16]:
dsir_out = dsir.run(fasta_path=fasta, mode="19nt",threshold=0.0, silent=True, override=True)

In [ ]:
dsir.show_results()

In [18]:
#CAREFULL WITH THE OVERRIDE ON THIS
ol_out = ol.run(fasta_path=fasta, override=False, silent=True)
#281m

In [ ]:
ol.show_results(1)

In [ ]:
dsir.show_results(fasta)

In [21]:
def get_dsir_commons(fasta: str):
    folders = dsir.show_results(fasta)
    all_js = []
    for x in folders:
        js = dsir.load_from_json((fasta, x.name))[0] #[0] porque retorna uma lista com os jsons, que no caso sempre só tem 1
        all_js.append(js)

    all_sets = []
    for j in all_js:
        cs = set()
        for s in j: #j is the json, which is a list of dicts
            cs.add(s["sirna"])
        all_sets.append(cs)

    common_sirnas = set.intersection(*all_sets)
    return common_sirnas

common_dsir = get_dsir_commons(fasta)



📂 Dsir MAPT_transcripts results
############################################################
Index  | Name
------------------------------------------------------------
1      | MAPT_transcripts_fasta_XM_005257370.5
2      | MAPT_transcripts_fasta_XM_054330116.1
3      | MAPT_transcripts_fasta_XM_054316136.1
4      | MAPT_transcripts_fasta_XM_054328567.1
5      | MAPT_transcripts_fasta_XM_054328571.1
6      | MAPT_transcripts_fasta_XM_047436075.1
7      | MAPT_transcripts_fasta_XM_054330112.1
8      | MAPT_transcripts_fasta_XM_054330113.1
9      | MAPT_transcripts_fasta_NM_001377267.1
10     | MAPT_transcripts_fasta_XM_054316131.1
11     | MAPT_transcripts_fasta_XM_054316143.1
12     | MAPT_transcripts_fasta_NM_001203251.2
13     | MAPT_transcripts_fasta_XM_054316132.1
14     | MAPT_transcripts_fasta_XM_054316144.1
15     | MAPT_transcripts_fasta_XM_047436080.1
16     | MAPT_transcripts_fasta_XM_054330118.1
17     | MAPT_transcripts_fasta_XM_005257362.5
18     | MAPT_transcripts_fasta_

In [22]:
def get_dsir_scores(fasta:str):
    folders = dsir.show_results(fasta)
    all_js = []
    for x in folders:
        js = dsir.load_from_json((fasta, x.name))[0] #[0] porque retorna uma lista com os jsons, que no caso sempre só tem 1
        all_js.append((x.name, js))

    all_sets = []
    for j in all_js:
        cs = set()
        for s in j[1]: #j[1] is the json, which is a list of dicts
            cs.add(s["sirna"])
        all_sets.append(cs)

    common_sirnas = set.intersection(*all_sets)
    common_scores = []

    for s in common_sirnas:
        out = {}
        out["sirna"] = s
        for j in all_js:
            for x in j[1]:
                if(x["sirna"] == s):
                    out[j[0]] = x["efficacy"]
        common_scores.append(out)
    return common_scores

common_dsir_scores = get_dsir_scores(fasta)
print(len(common_dsir_scores))


📂 Dsir MAPT_transcripts results
############################################################
Index  | Name
------------------------------------------------------------
1      | MAPT_transcripts_fasta_XM_005257370.5
2      | MAPT_transcripts_fasta_XM_054330116.1
3      | MAPT_transcripts_fasta_XM_054316136.1
4      | MAPT_transcripts_fasta_XM_054328567.1
5      | MAPT_transcripts_fasta_XM_054328571.1
6      | MAPT_transcripts_fasta_XM_047436075.1
7      | MAPT_transcripts_fasta_XM_054330112.1
8      | MAPT_transcripts_fasta_XM_054330113.1
9      | MAPT_transcripts_fasta_NM_001377267.1
10     | MAPT_transcripts_fasta_XM_054316131.1
11     | MAPT_transcripts_fasta_XM_054316143.1
12     | MAPT_transcripts_fasta_NM_001203251.2
13     | MAPT_transcripts_fasta_XM_054316132.1
14     | MAPT_transcripts_fasta_XM_054316144.1
15     | MAPT_transcripts_fasta_XM_047436080.1
16     | MAPT_transcripts_fasta_XM_054330118.1
17     | MAPT_transcripts_fasta_XM_005257362.5
18     | MAPT_transcripts_fasta_

In [ ]:
df_dsir = pd.DataFrame(common_dsir_scores)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)          # avoids line wrapping
pd.set_option('display.max_colwidth', None)   # full cell content
df_dsir["min"] = df_dsir.iloc[:, 1:].min(axis=1)
df_dsir["mean"] = df_dsir.iloc[:, 1:].mean(axis=1)
df_dsir["std"] = df_dsir.iloc[:, 1:].std(axis=1)

df_dsir["score1"] = (
    df_dsir["mean"]
    - 0.5 * df_dsir["std"]      # penalize inconsistency
    + 0.5 * df_dsir["min"]      # reward strong worst-case
)
df_dsir = df_dsir.sort_values("score1", ascending=False)
display(df_dsir)
df_dsir.to_csv("dsir_sirna_scores.csv", index=False)

In [ ]:
def get_ol_commons(fasta: str, ranked: bool = True, filtered: bool = True):
    folders = ol.show_results(fasta)
    all_js = []
    for x in folders:
        js = ol.load_oligoformer_json((fasta, x.name), ranked=ranked, filtered=filtered)[0] #[0] porque retorna uma lista com os jsons, que no caso sempre só tem 1
        all_js.append(js)

    all_sets = []
    for j in all_js:
        cs = set()
        for s in j: #j is the json, which is a list of dicts
            cs.add(s["sirna"])
        all_sets.append(cs)

    common_sirnas = set.intersection(*all_sets)
    return common_sirnas

common_ol = get_ol_commons(fasta)

In [ ]:
print(common_ol)
print(len(common_ol))

In [ ]:
def get_ol_scores(fasta:str, ranked: bool = True, filtered: bool = True):
    folders = ol.show_results(fasta)
    all_js = []
    for x in folders:
        js = ol.load_oligoformer_json((fasta, x.name), ranked=ranked, filtered=filtered)[0] #[0] porque retorna uma lista com os jsons, que no caso sempre só tem 1
        all_js.append(js)

    all_sets = []
    for j in all_js:
        cs = set()
        for s in j: #j is the json, which is a list of dicts
            cs.add(s["sirna"])
        all_sets.append(cs)

    common_sirnas = set.intersection(*all_sets)
    common_scores = []

    for s in common_sirnas:
        out = {}
        out["sirna"] = s
        for j in all_js:
            for x in j:
                if(x["sirna"] == s):
                    out[x["origin_file"]] = x["efficacy"]
        common_scores.append(out)
    return common_scores

common_ol_scores = get_ol_scores(fasta)


In [ ]:
pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')
pd.reset_option('display.width')
pd.reset_option('display.max_colwidth')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)          # avoids line wrapping
pd.set_option('display.max_colwidth', None)   # full cell content
df_ol = pd.DataFrame(common_ol_scores)
df_ol["min"] = df_ol.iloc[:, 1:].min(axis=1)
df_ol["mean"] = df_ol.iloc[:, 1:].mean(axis=1)
df_ol["std"] = df_ol.iloc[:, 1:].std(axis=1)

df_ol["score1"] = (
    df_ol["mean"]
    - 0.5 * df_ol["std"]      # penalize inconsistency
    + 0.5 * df_ol["min"]      # reward strong worst-case
)
df_ol = df_ol.sort_values("score1", ascending=False)
display(df_ol)
df_ol.to_csv("ol_sirna_scores.csv", index=False)

In [ ]:
nm_df = df_ol.loc[:, ["sirna"] + [col for col in df_ol.columns if "NM" in col]]
display(nm_df)

In [ ]:
nm_values = nm_df.iloc[:, 1:]  # exclude sirna column
nm_df["min"]  = nm_values.min(axis=1)
nm_df["mean"] = nm_values.mean(axis=1)
nm_df["std"]  = nm_values.std(axis=1)

nm_df["score"] = (
    nm_df["mean"]
    - 0.5 * nm_df["std"]
    + 0.5 * nm_df["min"]
)
display(nm_df)
nm_df.to_csv("ol_NM_sirna_scores.csv", index=False)

In [ ]:
def jaccard_index(df1, df2, col="sirna"):
    set1 = set(df1[col])
    set2 = set(df2[col])
    
    intersection = set1 & set2
    union = set1 | set2
    
    if len(union) == 0:
        return 0
    
    return len(intersection) / len(union)

In [ ]:
J = jaccard_index(df_dsir, df_ol)
print(f"Jaccard Index: {J:.4f}")

In [ ]:
def jaccard_details(df1, df2, col="sirna"):
    set1 = set(df1[col])
    set2 = set(df2[col])
    
    return {
        "intersection": set1 & set2,
        "only_in_df1": set1 - set2,
        "only_in_df2": set2 - set1
    }

In [ ]:
details = jaccard_details(df_dsir, df_ol)

print("Common siRNAs:", len(details["intersection"]))
print("Only DSIR:", len(details["only_in_df1"]))
print("Only OligoFormer:", len(details["only_in_df2"]))

In [ ]:
def overlap_coefficient(df1, df2, col="sirna"):
    set1 = set(df1[col])
    set2 = set(df2[col])
    
    intersection = set1 & set2
    
    if min(len(set1), len(set2)) == 0:
        return 0
    
    return len(intersection) / min(len(set1), len(set2))


def dice_coefficient(df1, df2, col="sirna"):
    set1 = set(df1[col])
    set2 = set(df2[col])
    
    intersection = set1 & set2
    
    if (len(set1) + len(set2)) == 0:
        return 0
    
    return 2 * len(intersection) / (len(set1) + len(set2))

In [ ]:
O = overlap_coefficient(df_dsir, df_ol)
D = dice_coefficient(df_dsir, df_ol)

print(f"Overlap coefficient: {O:.4f}")
print(f"Dice coefficient: {D:.4f}")

In [ ]:
def top_n_overlap(df1, df2, n=10, col="sirna", score_col="score1"):
    top1 = set(df1.sort_values(score_col, ascending=False).head(n)[col])
    top2 = set(df2.sort_values(score_col, ascending=False).head(n)[col])
    
    intersection = top1 & top2
    
    return {
        "n": n,
        "intersection": intersection,
        "count": len(intersection),
        "fraction": len(intersection) / n if n > 0 else 0,
        "only_in_df1": top1 - top2,
        "only_in_df2": top2 - top1,
    }

In [ ]:
for n in [5, 10, 20]:
    res = top_n_overlap(df_dsir, df_ol, n=n)
    print(f"Top-{n}: {res['count']}/{n} em comum ({res['fraction']:.1%})")

In [ ]:
from scipy.stats import spearmanr, kendalltau

def paired_scores(df1, df2, col="sirna", score_col="score1", name1="dsir", name2="ol"):
    common = set(df1[col]) & set(df2[col])
    
    s1 = df1[df1[col].isin(common)][[col, score_col]].rename(columns={score_col: name1})
    s2 = df2[df2[col].isin(common)][[col, score_col]].rename(columns={score_col: name2})
    
    merged = s1.merge(s2, on=col)
    return merged


def rank_correlation(df1, df2, col="sirna", score_col="score1", name1="dsir", name2="ol"):
    merged = paired_scores(df1, df2, col, score_col, name1, name2)
    
    rho, p_spearman = spearmanr(merged[name1], merged[name2])
    tau, p_kendall = kendalltau(merged[name1], merged[name2])
    
    return {
        "n_common": len(merged),
        "spearman_rho": rho,
        "spearman_p": p_spearman,
        "kendall_tau": tau,
        "kendall_p": p_kendall,
        "merged": merged,
    }

In [ ]:
corr = rank_correlation(df_dsir, df_ol)

print(f"N siRNAs comuns: {corr['n_common']}")
print(f"Spearman rho: {corr['spearman_rho']:.4f} (p={corr['spearman_p']:.4g})")
print(f"Kendall tau: {corr['kendall_tau']:.4f} (p={corr['kendall_p']:.4g})")

display(corr["merged"].sort_values("dsir", ascending=False))

In [ ]:
from sklearn.metrics import cohen_kappa_score, confusion_matrix

def binary_agreement_quantile(df1, df2, quantile=0.5, col="sirna", score_col="score1", name1="dsir", name2="ol"):
    merged = paired_scores(df1, df2, col, score_col, name1, name2)
    
    t1 = merged[name1].quantile(quantile)
    t2 = merged[name2].quantile(quantile)
    
    merged[f"{name1}_bin"] = (merged[name1] >= t1).astype(int)
    merged[f"{name2}_bin"] = (merged[name2] >= t2).astype(int)
    
    kappa = cohen_kappa_score(merged[f"{name1}_bin"], merged[f"{name2}_bin"], labels=[0, 1])
    cm = confusion_matrix(merged[f"{name1}_bin"], merged[f"{name2}_bin"], labels=[1, 0])
    
    return {
        "merged": merged,
        "threshold_1": t1,
        "threshold_2": t2,
        "kappa": kappa,
        "confusion_matrix": cm,
    }

## OligoFormer sem filtro — comparação com DSIR

O `df_ol` construído acima usa `filtered=True` (default de `get_ol_commons`/`get_ol_scores`), ou seja,
já passou pela etapa de filtragem do OligoFormer. Aqui construímos a versão **sem filtro**
(`filtered=False`) do mesmo jeito, sem rodar o modelo de novo — só recarregando o output já salvo em
disco — e repetimos exatamente a mesma bateria de métricas usada acima para comparar com o DSIR.

In [ ]:
common_ol_unfiltered = get_ol_commons(fasta, filtered=False)
common_ol_scores_unfiltered = get_ol_scores(fasta, filtered=False)

df_ol_unfiltered = pd.DataFrame(common_ol_scores_unfiltered)
df_ol_unfiltered["min"] = df_ol_unfiltered.iloc[:, 1:].min(axis=1)
df_ol_unfiltered["mean"] = df_ol_unfiltered.iloc[:, 1:].mean(axis=1)
df_ol_unfiltered["std"] = df_ol_unfiltered.iloc[:, 1:].std(axis=1)

df_ol_unfiltered["score1"] = (
    df_ol_unfiltered["mean"]
    - 0.5 * df_ol_unfiltered["std"]      # penalize inconsistency
    + 0.5 * df_ol_unfiltered["min"]      # reward strong worst-case
)
df_ol_unfiltered = df_ol_unfiltered.sort_values("score1", ascending=False)
display(df_ol_unfiltered)
df_ol_unfiltered.to_csv("ol_sirna_scores_unfiltered.csv", index=False)


In [ ]:
J_unf = jaccard_index(df_dsir, df_ol_unfiltered)
details_unf = jaccard_details(df_dsir, df_ol_unfiltered)

print(f"Jaccard Index (DSIR vs OL sem filtro): {J_unf:.4f}")
print("Common siRNAs:", len(details_unf["intersection"]))
print("Only DSIR:", len(details_unf["only_in_df1"]))
print("Only OligoFormer (sem filtro):", len(details_unf["only_in_df2"]))


In [ ]:
O_unf = overlap_coefficient(df_dsir, df_ol_unfiltered)
D_unf = dice_coefficient(df_dsir, df_ol_unfiltered)

print(f"Overlap coefficient: {O_unf:.4f}")
print(f"Dice coefficient: {D_unf:.4f}")


In [ ]:
top_n_results_unf = []
for n in [5, 10, 20]:
    r = top_n_overlap(df_dsir, df_ol_unfiltered, n=n)
    top_n_results_unf.append(r)
    print(f"Top-{n}: {r['count']}/{n} em comum ({r['fraction']:.1%})")


In [ ]:
corr_unf = rank_correlation(df_dsir, df_ol_unfiltered)

print(f"N siRNAs comuns: {corr_unf['n_common']}")
print(f"Spearman rho: {corr_unf['spearman_rho']:.4f} (p={corr_unf['spearman_p']:.4g})")
print(f"Kendall tau: {corr_unf['kendall_tau']:.4f} (p={corr_unf['kendall_p']:.4g})")

display(corr_unf["merged"].sort_values("dsir", ascending=False))


In [ ]:
import os
os.makedirs("final_results", exist_ok=True)

# Salva todos os resultados dos testes comparativos em um arquivo
import json
import numpy as np

def _serialize(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, pd.DataFrame):
        return obj.to_dict(orient="records")
    if isinstance(obj, pd.Series):
        return obj.to_dict()
    if isinstance(obj, set):
        return sorted(obj)
    raise TypeError(f"Tipo não serializável: {type(obj)}")

top_n_results = [top_n_overlap(df_dsir, df_ol, n=n) for n in [5, 10, 20]]

def _metrics_block(J, details, O, D, top_n_res, corr):
    return {
        "jaccard_index": J,
        "jaccard_details": {
            "n_intersection": len(details["intersection"]),
            "n_only_dsir": len(details["only_in_df1"]),
            "n_only_ol": len(details["only_in_df2"]),
        },
        "overlap_coefficient": O,
        "dice_coefficient": D,
        "top_n_overlap": [
            {"n": r["n"], "count": r["count"], "fraction": r["fraction"]}
            for r in top_n_res
        ],
        "rank_correlation": {
            "n_common": corr["n_common"],
            "spearman_rho": corr["spearman_rho"],
            "spearman_p": corr["spearman_p"],
            "kendall_tau": corr["kendall_tau"],
            "kendall_p": corr["kendall_p"],
        },
    }

resultados_finais = {
    "oligoformer_filtrado": _metrics_block(J, details, O, D, top_n_results, corr),
    "oligoformer_sem_filtro": _metrics_block(J_unf, details_unf, O_unf, D_unf, top_n_results_unf, corr_unf),
}

output_path = "final_results/resultados_MAPT.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(resultados_finais, f, indent=2, ensure_ascii=False, default=_serialize)

print(f"Resultados salvos em {output_path}")


## Concordância com o DSIR: OligoFormer filtrado vs sem filtro

Gráfico de barras agrupadas comparando as 4 métricas de concordância (Jaccard, Overlap, Dice,
Spearman rho) entre as duas versões do OligoFormer, lado a lado.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

metric_labels = ["Jaccard", "Overlap", "Dice", "Spearman rho", "Kendall tau"]
valores_filtrado = [J, O, D, corr["spearman_rho"], corr["kendall_tau"]]
valores_sem_filtro = [J_unf, O_unf, D_unf, corr_unf["spearman_rho"], corr_unf["kendall_tau"]]

x = np.arange(len(metric_labels))
largura = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - largura/2, valores_filtrado, largura, label="OligoFormer (filtrado)")
ax.bar(x + largura/2, valores_sem_filtro, largura, label="OligoFormer (sem filtro)")

ax.set_ylabel("Valor da métrica")
ax.set_title("Concordância com o DSIR: OligoFormer filtrado vs sem filtro")
ax.set_xticks(x)
ax.set_xticklabels(metric_labels)
ax.axhline(0, color="black", linewidth=0.8)
ax.legend()

for i, (vf, vs) in enumerate(zip(valores_filtrado, valores_sem_filtro)):
    ax.text(i - largura/2, vf + (0.02 if vf >= 0 else -0.05), f"{vf:.3f}", ha="center", fontsize=9)
    ax.text(i + largura/2, vs + (0.02 if vs >= 0 else -0.05), f"{vs:.3f}", ha="center", fontsize=9)

plt.tight_layout()
plt.show()


## Conjuntos de siRNAs: OligoFormer sem filtro, filtrado e DSIR

Como o filtro do OligoFormer só remove candidatos (nunca adiciona), `OL(filtrado)` é sempre um
subconjunto de `OL(sem filtro)` — por isso um diagrama de Venn de 3 conjuntos "genérico" seria
enganoso aqui (ele sugere três grupos independentes, quando na real um contém o outro). Usamos o
`venn3` mesmo assim, mas com os conjuntos reais: como a região "só no OL filtrado" é sempre vazia
por construção, o diagrama acaba refletindo essa contenção corretamente. Vale notar que a área dos
círculos do `matplotlib-venn` é só aproximadamente proporcional às contagens — os números em cada
região são exatos, o tamanho visual do círculo não é.

Ao lado, isolamos a pergunta mais interessante: dos siRNAs que o filtro **excluiu** do OligoFormer,
quantos o DSIR também "rejeitou" (não estão no conjunto do DSIR) vs quantos o DSIR escolheu de
qualquer forma — ou seja, se o filtro do OL concorda ou não com o que o DSIR considera relevante.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib_venn import venn3, venn2

set_ol = set(df_ol["sirna"])
set_ol_unf = set(df_ol_unfiltered["sirna"])
set_dsir = set(df_dsir["sirna"])

assert set_ol.issubset(set_ol_unf), "Esperado que o conjunto filtrado seja subconjunto do sem filtro"
removidos_pelo_filtro = set_ol_unf - set_ol

print(f"OligoFormer sem filtro: {len(set_ol_unf)} siRNAs")
print(f"OligoFormer filtrado:   {len(set_ol)} siRNAs")
print(f"Removidos pelo filtro:  {len(removidos_pelo_filtro)} siRNAs")
print(f"Desses removidos, também presentes no DSIR: {len(removidos_pelo_filtro & set_dsir)}")

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

venn3(
    [set_ol_unf, set_ol, set_dsir],
    set_labels=("OligoFormer (sem filtro)", "OligoFormer (filtrado)", "DSIR"),
    ax=axes[0],
)
axes[0].set_title("Visão geral: OL sem filtro, OL filtrado e DSIR")

venn2(
    [removidos_pelo_filtro, set_dsir],
    set_labels=("Excluídos pelo filtro (OL)", "DSIR"),
    ax=axes[1],
)
axes[1].set_title("O que o filtro excluiu vs. o que o DSIR escolheu")

plt.tight_layout()
plt.show()


ALLNYLAM - modificações químicas
dados de validação


# Validação com dataset de patente — PCSK9 (NM_174936.3)

Esta seção roda o mesmo pipeline usado para o MAPT (DSIR + OligoFormer), mas agora sobre o transcrito
`NM_174936.3` (PCSK9), que é o único transcrito descrito na patente de onde veio
`pcsk9_validation_dataset_ranked.json`. Depois, os resultados dos dois modelos são validados contra
esse dataset, que contém 175 siRNAs com dados experimentais reais de silenciamento (não apenas scores
preditos por outro modelo).

Diferença importante em relação ao MAPT: como aqui há apenas **um** transcrito, os dataframes de saída
do DSIR/OligoFormer terão apenas uma coluna de score (não uma por transcrito), então a etapa de filtrar
colunas "NM" (feita para o MAPT) não é necessária aqui.

## 1. Obtenção do transcrito PCSK9 (NM_174936.3)

In [ ]:
# O transcrito já é conhecido diretamente a partir do dataset da patente (NM_174936.3, PCSK9 humano),
# então buscamos a FASTA diretamente pela accession, sem precisar resolver symbol -> gene_id -> uid.
pcsk9_accession = "NM_174936.3"
pcsk9_alias = "PCSK9"

fasta_pcsk9 = get_fasta_from_accession(pcsk9_accession, alias=pcsk9_alias, absolute_alias=False)
print("FASTA salva em:", fasta_pcsk9)

## 2. Carregando o dataset de validação da patente

In [ ]:
with open("pcsk9_validation_dataset_ranked.json", "r", encoding="utf-8") as f:
    patent_data = json.load(f)

df_patent = pd.DataFrame(patent_data)
print("Total de siRNAs na patente:", len(df_patent))
display(df_patent.head())

## 3. Rodando os modelos

Roda os três modelos sobre o transcrito PCSK9. A partir desta reformulação, a única versão do OligoFormer usada no experimento é o batch de 10 runs (não há mais uma run "solo" separada).

### 3.1 DSIR — modo 19nt

In [ ]:
# threshold baixado de 0.9 para 0.0: com 0.9 o DSIR retornava so 22 candidatos pra esse transcrito,
# o que deixava a validacao contra a patente (175 siRNAs) e a comparacao com o OligoFormer (1351
# candidatos) com amostra pequena demais pra ter poder estatistico. Com threshold=0.0 o DSIR
# pontua praticamente todas as janelas possiveis do transcrito, e o filtro/ranking fica por nossa
# conta nas celulas de analise (nao precisamos do corte do proprio DSIR).
# override=True aqui e proposital: a rodada anterior (threshold=0.9) ficou salva em disco, e com
# override=False o dsir reaproveitaria esse resultado antigo em vez de recalcular com o
# threshold novo (foi exatamente isso que aconteceu da primeira vez que baixamos o threshold).
dsir_out_pcsk9 = dsir.run(fasta_path=fasta_pcsk9, mode="19nt", threshold=0.0, silent=True, override=True)
dsir.show_results()

### 3.2 DSIR — modo 21nt

In [ ]:
import shutil as _shutil

fasta_pcsk9_21nt = "PCSK9_NM_174936.3_21nt.fasta"
_shutil.copy(fasta_pcsk9, fasta_pcsk9_21nt)

dsir_out_pcsk9_21nt = dsir.run(fasta_path=fasta_pcsk9_21nt, mode="21nt", threshold=0.0, silent=True, override=True)
dsir.show_results()

### 3.3 OligoFormer — batch de 10 runs

In [ ]:
import pickle
from pathlib import Path

CHECKPOINT_FILE = Path("pcsk9_batch_checkpoint.pkl")

if CHECKPOINT_FILE.exists():
    with open(CHECKPOINT_FILE, "rb") as f:
        batch_raw_results = pickle.load(f)

    print(f"Checkpoint loaded: {len(batch_raw_results)} runs already completed.")
else:
    batch_raw_results = []
    print("No checkpoint found. Starting from 0 runs.")

In [ ]:
import shutil as _shutil

fasta_pcsk9_batch = "PCSK9_NM_174936.3_batch.fasta"
_shutil.copy(fasta_pcsk9, fasta_pcsk9_batch)

N_BATCH_RUNS = 10

for i in range(len(batch_raw_results), N_BATCH_RUNS):
    
    ol.run(fasta_path=fasta_pcsk9_batch, override=True, silent=True)
    folders = ol.show_results(fasta_pcsk9_batch)
    assert len(folders) == 1, f"Esperado 1 transcrito, encontrado {len(folders)}"

    raw = ol.load_oligoformer_json(
        (fasta_pcsk9_batch, folders[0].name),
        ranked=True,
        filtered=False
    )[0]

    batch_raw_results.append(raw)

    # Save after EVERY completed run
    with open(CHECKPOINT_FILE, "wb") as f:
        pickle.dump(batch_raw_results, f)

    print(
        f"Run {i + 1}/{N_BATCH_RUNS}: "
        f"{len(raw)} candidatos capturados"
    )

## 4. Construção dos dataframes

### 4.1 DSIR — 19nt

In [ ]:
common_dsir_pcsk9 = get_dsir_commons(fasta_pcsk9)
common_dsir_scores_pcsk9 = get_dsir_scores(fasta_pcsk9)
print("siRNAs (DSIR) encontrados para PCSK9:", len(common_dsir_scores_pcsk9))

df_dsir_pcsk9 = pd.DataFrame(common_dsir_scores_pcsk9)
df_dsir_pcsk9["min"] = df_dsir_pcsk9.iloc[:, 1:].min(axis=1)
df_dsir_pcsk9["mean"] = df_dsir_pcsk9.iloc[:, 1:].mean(axis=1)
df_dsir_pcsk9["std"] = df_dsir_pcsk9.iloc[:, 1:].std(axis=1)

df_dsir_pcsk9["score1"] = (
    df_dsir_pcsk9["mean"]
    - 0.5 * df_dsir_pcsk9["std"]
    + 0.5 * df_dsir_pcsk9["min"]
)
df_dsir_pcsk9 = df_dsir_pcsk9.sort_values("score1", ascending=False)
display(df_dsir_pcsk9)
df_dsir_pcsk9.to_csv("dsir_pcsk9_sirna_scores.csv", index=False)

### 4.2 DSIR — 21nt

In [ ]:
common_dsir21_pcsk9 = get_dsir_commons(fasta_pcsk9_21nt)
common_dsir21_scores_pcsk9 = get_dsir_scores(fasta_pcsk9_21nt)
print("siRNAs (DSIR 21nt) encontrados para PCSK9:", len(common_dsir21_scores_pcsk9))

df_dsir21_pcsk9 = pd.DataFrame(common_dsir21_scores_pcsk9)
df_dsir21_pcsk9["min"] = df_dsir21_pcsk9.iloc[:, 1:].min(axis=1)
df_dsir21_pcsk9["mean"] = df_dsir21_pcsk9.iloc[:, 1:].mean(axis=1)
df_dsir21_pcsk9["std"] = df_dsir21_pcsk9.iloc[:, 1:].std(axis=1)

df_dsir21_pcsk9["score1"] = (
    df_dsir21_pcsk9["mean"]
    - 0.5 * df_dsir21_pcsk9["std"]
    + 0.5 * df_dsir21_pcsk9["min"]
)
df_dsir21_pcsk9 = df_dsir21_pcsk9.sort_values("score1", ascending=False)
display(df_dsir21_pcsk9)
df_dsir21_pcsk9.to_csv("dsir21_pcsk9_sirna_scores.csv", index=False)

### 4.3 OligoFormer — batch (agregado)

In [ ]:
from collections import defaultdict

scores_by_sirna = defaultdict(list)
mrna_by_sirna = {}
position_by_sirna = {}

for raw in batch_raw_results:
    for entry in raw:
        s = entry["sirna"]
        scores_by_sirna[s].append(entry["efficacy"])
        mrna_by_sirna.setdefault(s, entry["mrna_segment"])
        position_by_sirna.setdefault(s, entry["position"])

batch_rows = []
for s, scores in scores_by_sirna.items():
    batch_rows.append({
        "sirna": s,
        "mrna_segment": mrna_by_sirna[s],
        "position": position_by_sirna[s],
        "n_runs_present": len(scores),
        "efficacy_mean": float(np.mean(scores)),
        "efficacy_std": float(np.std(scores)),
        "efficacy_min": float(np.min(scores)),
        "efficacy_max": float(np.max(scores)),
    })

df_ol_batch = pd.DataFrame(batch_rows).sort_values("efficacy_mean", ascending=False)

print(f"siRNAs únicos vistos em pelo menos 1 dos {N_BATCH_RUNS} runs: {len(df_ol_batch)}")
print(f"siRNAs presentes em TODOS os {N_BATCH_RUNS} runs: {(df_ol_batch['n_runs_present'] == N_BATCH_RUNS).sum()}")
print(f"Desvio-padrão médio da eficácia (entre runs, por siRNA): {df_ol_batch['efficacy_std'].mean():.4f}")
display(df_ol_batch.head(20))

### 4.4 OligoFormer — batch filtrado (agregado)

Reaproveita o mesmo `batch_raw_results` carregado do checkpoint pickle (10 runs, `filtered=False`), mas descarta em memória toda entrada com `filters["filter"] != 0` antes de agregar — sem rodar o OligoFormer de novo. Resultado equivalente ao `df_ol_batch` (seção 4.3), só que na versão filtrada.

In [ ]:
from collections import defaultdict

scores_by_sirna_f = defaultdict(list)
mrna_by_sirna_f = {}
position_by_sirna_f = {}

for raw in batch_raw_results:
    for entry in raw:
        if entry["filters"]["filter"] != 0:
            continue  # descarta candidatos que o OligoFormer marcou como filtrados
        s = entry["sirna"]
        scores_by_sirna_f[s].append(entry["efficacy"])
        mrna_by_sirna_f.setdefault(s, entry["mrna_segment"])
        position_by_sirna_f.setdefault(s, entry["position"])

batch_rows_f = []
for s, scores in scores_by_sirna_f.items():
    batch_rows_f.append({
        "sirna": s,
        "mrna_segment": mrna_by_sirna_f[s],
        "position": position_by_sirna_f[s],
        "n_runs_present": len(scores),
        "efficacy_mean": float(np.mean(scores)),
        "efficacy_std": float(np.std(scores)),
        "efficacy_min": float(np.min(scores)),
        "efficacy_max": float(np.max(scores)),
    })

df_ol_batch_filtered = pd.DataFrame(batch_rows_f).sort_values("efficacy_mean", ascending=False)

print(f"siRNAs únicos (filtrados, filter==0) vistos em pelo menos 1 dos {N_BATCH_RUNS} runs: {len(df_ol_batch_filtered)}")
print(f"siRNAs presentes em TODOS os {N_BATCH_RUNS} runs: {(df_ol_batch_filtered['n_runs_present'] == N_BATCH_RUNS).sum()}")
print(f"Desvio-padrão médio da eficácia (entre runs, por siRNA): {df_ol_batch_filtered['efficacy_std'].mean():.4f}")
display(df_ol_batch_filtered.head(20))


## 5. Métricas de concordância: OligoFormer (batch) vs DSIR-19nt (PCSK9)

Reaproveita exatamente a mesma bateria de métricas definida e usada na seção do MAPT (`jaccard_index`, `jaccard_details`, `overlap_coefficient`, `dice_coefficient`, `top_n_overlap`, `rank_correlation`), agora aplicada duas vezes para o PCSK9: primeiro `df_dsir_pcsk9` (DSIR-19nt, seção 4.1) contra `df_ol_batch` (OligoFormer batch sem filtro, seção 4.3), depois contra `df_ol_batch_filtered` (OligoFormer batch filtrado, seção 4.4).

Como `df_ol_batch`/`df_ol_batch_filtered` guardam `efficacy_mean`/`efficacy_std`/`efficacy_min` em vez de `score1`, primeiro adicionamos a coluna `score1` a cada um usando a mesma fórmula já usada em todos os outros dataframes do notebook (`mean - 0.5*std + 0.5*min`), só assim `top_n_overlap`/`rank_correlation` (que usam `score1` por padrão) funcionam sem precisar alterar as funções.

In [ ]:
df_ol_batch["score1"] = (
    df_ol_batch["efficacy_mean"]
    - 0.5 * df_ol_batch["efficacy_std"]
    + 0.5 * df_ol_batch["efficacy_min"]
)
df_ol_batch_filtered["score1"] = (
    df_ol_batch_filtered["efficacy_mean"]
    - 0.5 * df_ol_batch_filtered["efficacy_std"]
    + 0.5 * df_ol_batch_filtered["efficacy_min"]
)


### 5.1 OligoFormer (batch, sem filtro) vs DSIR-19nt

In [ ]:
J_ol_batch_pcsk9 = jaccard_index(df_dsir_pcsk9, df_ol_batch)
details_ol_batch_pcsk9 = jaccard_details(df_dsir_pcsk9, df_ol_batch)

print(f"Jaccard Index (DSIR-19nt vs OL batch sem filtro): {J_ol_batch_pcsk9:.4f}")
print("Common siRNAs:", len(details_ol_batch_pcsk9["intersection"]))
print("Only DSIR:", len(details_ol_batch_pcsk9["only_in_df1"]))
print("Only OligoFormer (batch sem filtro):", len(details_ol_batch_pcsk9["only_in_df2"]))


In [ ]:
O_ol_batch_pcsk9 = overlap_coefficient(df_dsir_pcsk9, df_ol_batch)
D_ol_batch_pcsk9 = dice_coefficient(df_dsir_pcsk9, df_ol_batch)

print(f"Overlap coefficient: {O_ol_batch_pcsk9:.4f}")
print(f"Dice coefficient: {D_ol_batch_pcsk9:.4f}")


In [ ]:
top_n_ol_batch_pcsk9 = []
for n in [5, 10, 20]:
    r = top_n_overlap(df_dsir_pcsk9, df_ol_batch, n=n)
    top_n_ol_batch_pcsk9.append(r)
    print(f"Top-{n}: {r['count']}/{n} em comum ({r['fraction']:.1%})")


In [ ]:
corr_ol_batch_pcsk9 = rank_correlation(df_dsir_pcsk9, df_ol_batch)

print(f"N siRNAs comuns: {corr_ol_batch_pcsk9['n_common']}")
print(f"Spearman rho: {corr_ol_batch_pcsk9['spearman_rho']:.4f} (p={corr_ol_batch_pcsk9['spearman_p']:.4g})")
print(f"Kendall tau: {corr_ol_batch_pcsk9['kendall_tau']:.4f} (p={corr_ol_batch_pcsk9['kendall_p']:.4g})")

display(corr_ol_batch_pcsk9["merged"].sort_values("dsir", ascending=False))


### 5.2 OligoFormer (batch, filtrado) vs DSIR-19nt

In [ ]:
J_ol_batch_f_pcsk9 = jaccard_index(df_dsir_pcsk9, df_ol_batch_filtered)
details_ol_batch_f_pcsk9 = jaccard_details(df_dsir_pcsk9, df_ol_batch_filtered)

print(f"Jaccard Index (DSIR-19nt vs OL batch filtrado): {J_ol_batch_f_pcsk9:.4f}")
print("Common siRNAs:", len(details_ol_batch_f_pcsk9["intersection"]))
print("Only DSIR:", len(details_ol_batch_f_pcsk9["only_in_df1"]))
print("Only OligoFormer (batch filtrado):", len(details_ol_batch_f_pcsk9["only_in_df2"]))


In [ ]:
O_ol_batch_f_pcsk9 = overlap_coefficient(df_dsir_pcsk9, df_ol_batch_filtered)
D_ol_batch_f_pcsk9 = dice_coefficient(df_dsir_pcsk9, df_ol_batch_filtered)

print(f"Overlap coefficient: {O_ol_batch_f_pcsk9:.4f}")
print(f"Dice coefficient: {D_ol_batch_f_pcsk9:.4f}")


In [ ]:
top_n_ol_batch_f_pcsk9 = []
for n in [5, 10, 20]:
    r = top_n_overlap(df_dsir_pcsk9, df_ol_batch_filtered, n=n)
    top_n_ol_batch_f_pcsk9.append(r)
    print(f"Top-{n}: {r['count']}/{n} em comum ({r['fraction']:.1%})")


In [ ]:
corr_ol_batch_f_pcsk9 = rank_correlation(df_dsir_pcsk9, df_ol_batch_filtered)

print(f"N siRNAs comuns: {corr_ol_batch_f_pcsk9['n_common']}")
print(f"Spearman rho: {corr_ol_batch_f_pcsk9['spearman_rho']:.4f} (p={corr_ol_batch_f_pcsk9['spearman_p']:.4g})")
print(f"Kendall tau: {corr_ol_batch_f_pcsk9['kendall_tau']:.4f} (p={corr_ol_batch_f_pcsk9['kendall_p']:.4g})")

display(corr_ol_batch_f_pcsk9["merged"].sort_values("dsir", ascending=False))


### 5.3 Concordância com o DSIR-19nt: OligoFormer batch sem filtro vs filtrado (PCSK9)

Mesmo gráfico de barras agrupadas usado na seção do MAPT, com as 5 métricas de concordância lado a lado para as duas versões do OligoFormer (batch).

In [ ]:
metric_labels_pcsk9 = ["Jaccard", "Overlap", "Dice", "Spearman rho", "Kendall tau"]
valores_ol_batch_pcsk9 = [
    J_ol_batch_pcsk9, O_ol_batch_pcsk9, D_ol_batch_pcsk9,
    corr_ol_batch_pcsk9["spearman_rho"], corr_ol_batch_pcsk9["kendall_tau"],
]
valores_ol_batch_f_pcsk9 = [
    J_ol_batch_f_pcsk9, O_ol_batch_f_pcsk9, D_ol_batch_f_pcsk9,
    corr_ol_batch_f_pcsk9["spearman_rho"], corr_ol_batch_f_pcsk9["kendall_tau"],
]

x = np.arange(len(metric_labels_pcsk9))
largura = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - largura/2, valores_ol_batch_pcsk9, largura, label="OligoFormer (batch, sem filtro)")
ax.bar(x + largura/2, valores_ol_batch_f_pcsk9, largura, label="OligoFormer (batch, filtrado)")

ax.set_ylabel("Valor da métrica")
ax.set_title("Concordância com o DSIR-19nt: OligoFormer batch sem filtro vs filtrado (PCSK9)")
ax.set_xticks(x)
ax.set_xticklabels(metric_labels_pcsk9)
ax.axhline(0, color="black", linewidth=0.8)
ax.legend()

for i, (vf, vs) in enumerate(zip(valores_ol_batch_pcsk9, valores_ol_batch_f_pcsk9)):
    ax.text(i - largura/2, vf + (0.02 if vf >= 0 else -0.05), f"{vf:.3f}", ha="center", fontsize=9)
    ax.text(i + largura/2, vs + (0.02 if vs >= 0 else -0.05), f"{vs:.3f}", ha="center", fontsize=9)

plt.tight_layout()
plt.show()


### 5.4 Guardando os resultados em um dict (ainda sem salvar em arquivo)

Reaproveita `_metrics_block` (já definida na seção do MAPT) para montar o mesmo formato de dicionário usado lá, com os dois blocos desta seção (OligoFormer batch sem filtro / filtrado vs DSIR-19nt). Fica só em memória por enquanto — a ideia é usar esse dict depois, junto com os outros resultados do PCSK9, na hora de montar o JSON de saída consolidado.

In [ ]:
resultados_concordancia_pcsk9_batch = {
    "oligoformer_batch_sem_filtro": _metrics_block(
        J_ol_batch_pcsk9, details_ol_batch_pcsk9, O_ol_batch_pcsk9, D_ol_batch_pcsk9,
        top_n_ol_batch_pcsk9, corr_ol_batch_pcsk9,
    ),
    "oligoformer_batch_filtrado": _metrics_block(
        J_ol_batch_f_pcsk9, details_ol_batch_f_pcsk9, O_ol_batch_f_pcsk9, D_ol_batch_f_pcsk9,
        top_n_ol_batch_f_pcsk9, corr_ol_batch_f_pcsk9,
    ),
}

resultados_concordancia_pcsk9_batch


## 6. Validação externa: comparação com dados experimentais da patente

Diferente da comparação DSIR-vs-OligoFormer (que compara dois *scores preditos* entre si), aqui
comparamos as predições dos modelos contra uma **medida de eficácia real** (percentual de mRNA
residual após transfecção, medido em 3 linhagens celulares × 2 concentrações). Valores **menores**
de `efficacy_score_10nM_avg`/`efficacy_score_0.1nM_avg` significam **mais silenciamento** (melhor
siRNA) — é o oposto da convenção dos scores dos modelos, onde valor **maior** = melhor.

Passos desta seção:
1. Carregar o dataset da patente (175 siRNAs).
2. Casar cada siRNA da patente com as saídas do DSIR e do OligoFormer.
3. Medir a cobertura: quantos dos 175 siRNAs da patente aparecem nos resultados de cada modelo.
4. Medir se a ordem prevista pelos modelos condiz com a ordem real de eficácia (correlação de rank).
5. Medir se os siRNAs mais eficazes de verdade aparecem no topo do ranking de cada modelo (top-N).

### 6.1 Separação por dose (10nM vs 0.1nM)

O `df_patent` traz, para cada siRNA, medidas de eficácia em duas concentrações
(`efficacy_score_10nM_avg` e `efficacy_score_0.1nM_avg`), além de `rank_by_10nM_avg`
(não existe um rank equivalente para 0.1nM no dataset). Para deixar as comparações de
cada dose independentes e evitar carregar colunas da dose errada nas seções seguintes,
separamos aqui em dois dataframes, cada um com as colunas comuns (identificação do
siRNA/duplex) mais as colunas específicas da sua dose.

In [ ]:
common_cols = [
    "Duplex Name", "Sense Trans Seq", "Antisense Trans Seq",
    "SEQ ID NO (Sense)", "SEQ ID NO (Antisense)",
    "Start In NM_174936.3", "End In NM_174936.3",
    "Table1 Source Page", "Table4 Source Page",
]

cols_10nM = ["Hela 10nM", "Hep3b 10nM", "HepG2 10nM", "efficacy_score_10nM_avg", "rank_by_10nM_avg"]
cols_0_1nM = ["Hela 0.1nM", "Hep3b 0.1nM", "HepG2 0.1nM", "efficacy_score_0.1nM_avg"]

df_patent_10nM = df_patent[common_cols + cols_10nM].copy()
df_patent_0_1nM = df_patent[common_cols + cols_0_1nM].copy()

print("df_patent_10nM:", df_patent_10nM.shape)
display(df_patent_10nM.head())

print("df_patent_0_1nM:", df_patent_0_1nM.shape)
display(df_patent_0_1nM.head())

### 6.2 Preparação: candidatos brutos por modelo (necessários para o matching por posição)

As funções `get_dsir_scores`/`get_ol_scores` (usadas nas seções 4.1–4.4) descartam os campos
`position`/`mrna_segment` de cada candidato, guardando só `sirna` + score agregado. Para o matching
com a patente (que é feito por posição, não por sequência — ver seção 6.3), precisamos recarregar os
dados brutos por transcrito, preservando esses campos, para os 4 modelos que serão validados nesta
seção: OligoFormer (batch, sem filtro), OligoFormer (batch, filtrado), DSIR-19nt e DSIR-21nt.


In [ ]:
def get_dsir_raw(fasta: str):
    """Retorna a lista bruta de candidatos do DSIR (sirna, mrna_segment, position, efficacy, ...)
    para um fasta de transcrito único, sem a etapa de interseção usada para múltiplos transcritos."""
    folders = dsir.show_results(fasta)
    assert len(folders) == 1, f"Esperado exatamente 1 transcrito, encontrado {len(folders)}"
    return dsir.load_from_json((fasta, folders[0].name))[0]


raw_dsir19_pcsk9 = get_dsir_raw(fasta_pcsk9)
raw_dsir21_pcsk9 = get_dsir_raw(fasta_pcsk9_21nt)

print("Candidatos brutos DSIR-19nt (PCSK9):", len(raw_dsir19_pcsk9))
print("Candidatos brutos DSIR-21nt (PCSK9):", len(raw_dsir21_pcsk9))


In [ ]:
raw_ol_pcsk9 = df_ol_batch.rename(columns={"efficacy_mean": "efficacy"})[
    ["sirna", "mrna_segment", "position", "efficacy"]
].to_dict(orient="records")

raw_ol_f_pcsk9 = df_ol_batch_filtered.rename(columns={"efficacy_mean": "efficacy"})[
    ["sirna", "mrna_segment", "position", "efficacy"]
].to_dict(orient="records")

print("Candidatos OligoFormer (batch, sem filtro) (PCSK9):", len(raw_ol_pcsk9))
print("Candidatos OligoFormer (batch, filtrado) (PCSK9):", len(raw_ol_f_pcsk9))


### 6.3 Matching por posição (patente ↔ modelos)

O campo `"sirna"` retornado pelo DSIR e pelo OligoFormer **não** é a fita senso — é a fita
**antisenso (guia)**, em 5'→3'. Cada entrada bruta também traz `"mrna_segment"` (a fita senso/alvo
correspondente, mesma orientação do mRNA) e `"position"`. Além disso, `"position"` bate **exatamente**
com a coluna `"Start In NM_174936.3"` da patente (mesmo sistema de coordenadas, 1-indexado) — testado
manualmente com um caso conhecido. Isso torna a posição uma chave de correspondência muito mais
confiável do que tentar casar sequências de tamanhos diferentes (19 nt vs 21 nt vs 23 nt) por
substring.

Estratégia adotada:
1. **Chave primária**: `position` do modelo == `Start In NM_174936.3` da patente.
2. **Confirmação cruzada (sanity check)**: verificar se o `sirna` do modelo (antisenso) aparece de
   fato como substring do `Antisense Trans Seq` da patente — serve como checagem de sanidade, não
   como critério de match em si.

Essa função é a base de todas as validações desta seção — é aplicada 1x por modelo em cada uma das
duas sub-seções de dose (10nM e 0.1nM).


In [ ]:
def match_patent_to_model_by_position(df_patent_dose, raw_model, position_tolerance=0):
    """
    Casa cada siRNA da patente (df_patent_dose, já filtrado pra uma dose) com uma entrada bruta do
    modelo usando a posição no transcrito como chave primária (confirmado empiricamente: position
    do modelo == Start In NM_174936.3 da patente). A checagem por sequência (sirna como substring
    do Antisense Trans Seq) é guardada à parte, como confirmação/sanidade, não como critério de match.

    position_tolerance permite folga de +/- N nt, caso a coordenada não bata exatamente para todas
    as entradas (por padrão 0, já que o teste manual mostrou correspondência exata).
    """
    position_index = {}
    for entry in raw_model:
        position_index.setdefault(entry["position"], []).append(entry)

    rows = []
    for _, row in df_patent_dose.iterrows():
        start = int(row["Start In NM_174936.3"])
        found = None
        for offset in range(-position_tolerance, position_tolerance + 1):
            candidates = position_index.get(start + offset)
            if candidates:
                found = candidates[0]
                break

        seq_confirmed = False
        if found is not None:
            antisense_norm = row["Antisense Trans Seq"].strip().upper().replace("T", "U")
            seq_confirmed = found["sirna"] in antisense_norm

        rows.append({
            "Duplex Name": row["Duplex Name"],
            "matched_sirna": found["sirna"] if found else None,
            "matched_mrna_segment": found["mrna_segment"] if found else None,
            "matched_position": found["position"] if found else None,
            "model_score": found["efficacy"] if found else None,
            "seq_confirmed": seq_confirmed,
        })

    matches_df = pd.DataFrame(rows)
    return df_patent_dose.merge(matches_df, on="Duplex Name", how="left")


## 6.4 Validação — dose 10nM

Valida os 4 modelos/variantes contra `df_patent_10nM` (seção 6.1), cada um passando pelos **3 C's**:

1. **Cobertura** — Jaccard, Overlap e Dice entre o conjunto de siRNAs da patente e o conjunto bruto
   de candidatos do modelo (a partir do matching por posição da seção 6.3).
2. **Correlação** — Spearman e Kendall entre o score do modelo e a eficácia real
   (`efficacy_score_10nM_avg`, sinal invertido para que maior = mais eficaz nos dois lados).
3. **Concordância (top-N pareado)** — interseção entre o top-N do próprio modelo e o top-N real
   (`rank_by_10nM_avg`), para N = 5, 10, 20, 50.

Os 4 pares avaliados: OligoFormer (batch, sem filtro), OligoFormer (batch, filtrado), DSIR-19nt e
DSIR-21nt — todos contra a mesma `df_patent_10nM`.


In [ ]:
def compare_with_ground_truth(matched_df, model_name):
    """Correlação (C de 'Correlação'): Spearman e Kendall entre o score do modelo e a eficácia real
    medida a 10nM (efficacy_score_10nM_avg, sinal invertido: maior = mais eficaz)."""
    valid = matched_df.dropna(subset=["model_score"]).copy()
    valid["real_effect"] = -valid["efficacy_score_10nM_avg"]

    if len(valid) < 3:
        print(f"[{model_name}] Poucos pares válidos ({len(valid)}) para calcular correlação.")
        return {"n": len(valid), "spearman_rho": None, "spearman_p": None, "kendall_tau": None, "kendall_p": None, "merged": valid}

    rho, p_rho = spearmanr(valid["model_score"], valid["real_effect"])
    tau, p_tau = kendalltau(valid["model_score"], valid["real_effect"])

    print(f"--- {model_name} vs eficácia real, 10nM (n={len(valid)}) ---")
    print(f"Spearman rho: {rho:.4f} (p={p_rho:.4g})")
    print(f"Kendall tau: {tau:.4f} (p={p_tau:.4g})")

    return {"n": len(valid), "spearman_rho": rho, "spearman_p": p_rho, "kendall_tau": tau, "kendall_p": p_tau, "merged": valid}


In [ ]:
def top_n_paired_agreement(matched_df, model_name, n_values=(5, 10, 20, 50)):
    """Concordância (C de 'Concordância'): compara o top-N do PRÓPRIO modelo (por model_score)
    contra o top-N real a 10nM (rank_by_10nM_avg), dentro do subconjunto de siRNAs casados com a
    patente. Mede se o modelo acerta QUEM são os melhores, não só se ele 'enxerga' essas posições."""
    valid = matched_df.dropna(subset=["model_score"]).copy()
    n_total = len(valid)

    by_model = valid.sort_values("model_score", ascending=False)
    by_real = valid.sort_values("rank_by_10nM_avg", ascending=True)

    results = []
    for n in n_values:
        if n > n_total:
            print(f"[{model_name}] Top-{n}: pulado (só {n_total} siRNAs casados no total)")
            continue

        top_model = set(by_model.head(n)["Duplex Name"])
        top_real = set(by_real.head(n)["Duplex Name"])
        overlap = top_model & top_real

        chance = (n * n) / n_total
        count = len(overlap)
        print(f"[{model_name}] Top-{n}: {count}/{n} do top do modelo também está no top real "
              f"({count/n:.1%}) | esperado por acaso: ~{chance:.1f}")

        results.append({
            "n": n,
            "overlap": count,
            "fraction": count / n,
            "chance_expected": chance,
            "duplex_names": sorted(overlap),
        })

    return results


#### 6.4.1 OligoFormer (batch, sem filtro) × 10nM


In [ ]:
patent_vs_ol_10nM = match_patent_to_model_by_position(df_patent_10nM, raw_ol_pcsk9)

n_matched_ol = patent_vs_ol_10nM["matched_sirna"].notna().sum()
n_confirmed_ol = patent_vs_ol_10nM["seq_confirmed"].sum()

print(f"OligoFormer (batch, sem filtro): {n_matched_ol}/{len(df_patent_10nM)} siRNAs da patente casados por posição "
      f"({n_matched_ol/len(df_patent_10nM):.1%}), {n_confirmed_ol} confirmados por sequência "
      f"({n_confirmed_ol/max(n_matched_ol,1):.1%})")


In [ ]:
n_raw_ol = len(raw_ol_pcsk9)
n_patent = len(df_patent_10nM)

jaccard_ol_10nM = n_matched_ol / (n_patent + n_raw_ol - n_matched_ol)
overlap_ol_10nM = n_matched_ol / min(n_patent, n_raw_ol)
dice_ol_10nM = 2 * n_matched_ol / (n_patent + n_raw_ol)

print(f"Cobertura — patente vs OligoFormer (batch, sem filtro):")
print(f"Jaccard: {jaccard_ol_10nM:.4f} | Overlap: {overlap_ol_10nM:.4f} | Dice: {dice_ol_10nM:.4f}")


In [ ]:
val_ol_10nM = compare_with_ground_truth(patent_vs_ol_10nM, "OligoFormer (batch, sem filtro)")


In [ ]:
top_n_ol_10nM = top_n_paired_agreement(patent_vs_ol_10nM, "OligoFormer (batch, sem filtro)")


#### 6.4.2 OligoFormer (batch, filtrado) × 10nM


In [ ]:
patent_vs_ol_f_10nM = match_patent_to_model_by_position(df_patent_10nM, raw_ol_f_pcsk9)

n_matched_ol_f = patent_vs_ol_f_10nM["matched_sirna"].notna().sum()
n_confirmed_ol_f = patent_vs_ol_f_10nM["seq_confirmed"].sum()

print(f"OligoFormer (batch, filtrado): {n_matched_ol_f}/{len(df_patent_10nM)} siRNAs da patente casados por posição "
      f"({n_matched_ol_f/len(df_patent_10nM):.1%}), {n_confirmed_ol_f} confirmados por sequência "
      f"({n_confirmed_ol_f/max(n_matched_ol_f,1):.1%})")


In [ ]:
n_raw_ol_f = len(raw_ol_f_pcsk9)
n_patent = len(df_patent_10nM)

jaccard_ol_f_10nM = n_matched_ol_f / (n_patent + n_raw_ol_f - n_matched_ol_f)
overlap_ol_f_10nM = n_matched_ol_f / min(n_patent, n_raw_ol_f)
dice_ol_f_10nM = 2 * n_matched_ol_f / (n_patent + n_raw_ol_f)

print(f"Cobertura — patente vs OligoFormer (batch, filtrado):")
print(f"Jaccard: {jaccard_ol_f_10nM:.4f} | Overlap: {overlap_ol_f_10nM:.4f} | Dice: {dice_ol_f_10nM:.4f}")


In [ ]:
val_ol_f_10nM = compare_with_ground_truth(patent_vs_ol_f_10nM, "OligoFormer (batch, filtrado)")


In [ ]:
top_n_ol_f_10nM = top_n_paired_agreement(patent_vs_ol_f_10nM, "OligoFormer (batch, filtrado)")


#### 6.4.3 DSIR-19nt × 10nM


In [ ]:
patent_vs_dsir19_10nM = match_patent_to_model_by_position(df_patent_10nM, raw_dsir19_pcsk9)

n_matched_dsir19 = patent_vs_dsir19_10nM["matched_sirna"].notna().sum()
n_confirmed_dsir19 = patent_vs_dsir19_10nM["seq_confirmed"].sum()

print(f"DSIR-19nt: {n_matched_dsir19}/{len(df_patent_10nM)} siRNAs da patente casados por posição "
      f"({n_matched_dsir19/len(df_patent_10nM):.1%}), {n_confirmed_dsir19} confirmados por sequência "
      f"({n_confirmed_dsir19/max(n_matched_dsir19,1):.1%})")


In [ ]:
n_raw_dsir19 = len(raw_dsir19_pcsk9)
n_patent = len(df_patent_10nM)

jaccard_dsir19_10nM = n_matched_dsir19 / (n_patent + n_raw_dsir19 - n_matched_dsir19)
overlap_dsir19_10nM = n_matched_dsir19 / min(n_patent, n_raw_dsir19)
dice_dsir19_10nM = 2 * n_matched_dsir19 / (n_patent + n_raw_dsir19)

print(f"Cobertura — patente vs DSIR-19nt:")
print(f"Jaccard: {jaccard_dsir19_10nM:.4f} | Overlap: {overlap_dsir19_10nM:.4f} | Dice: {dice_dsir19_10nM:.4f}")


In [ ]:
val_dsir19_10nM = compare_with_ground_truth(patent_vs_dsir19_10nM, "DSIR-19nt")


In [ ]:
top_n_dsir19_10nM = top_n_paired_agreement(patent_vs_dsir19_10nM, "DSIR-19nt")


#### 6.4.4 DSIR-21nt × 10nM


In [ ]:
patent_vs_dsir21_10nM = match_patent_to_model_by_position(df_patent_10nM, raw_dsir21_pcsk9)

n_matched_dsir21 = patent_vs_dsir21_10nM["matched_sirna"].notna().sum()
n_confirmed_dsir21 = patent_vs_dsir21_10nM["seq_confirmed"].sum()

print(f"DSIR-21nt: {n_matched_dsir21}/{len(df_patent_10nM)} siRNAs da patente casados por posição "
      f"({n_matched_dsir21/len(df_patent_10nM):.1%}), {n_confirmed_dsir21} confirmados por sequência "
      f"({n_confirmed_dsir21/max(n_matched_dsir21,1):.1%})")


In [ ]:
n_raw_dsir21 = len(raw_dsir21_pcsk9)
n_patent = len(df_patent_10nM)

jaccard_dsir21_10nM = n_matched_dsir21 / (n_patent + n_raw_dsir21 - n_matched_dsir21)
overlap_dsir21_10nM = n_matched_dsir21 / min(n_patent, n_raw_dsir21)
dice_dsir21_10nM = 2 * n_matched_dsir21 / (n_patent + n_raw_dsir21)

print(f"Cobertura — patente vs DSIR-21nt:")
print(f"Jaccard: {jaccard_dsir21_10nM:.4f} | Overlap: {overlap_dsir21_10nM:.4f} | Dice: {dice_dsir21_10nM:.4f}")


In [ ]:
val_dsir21_10nM = compare_with_ground_truth(patent_vs_dsir21_10nM, "DSIR-21nt")


In [ ]:
top_n_dsir21_10nM = top_n_paired_agreement(patent_vs_dsir21_10nM, "DSIR-21nt")


## 6.5 Validação — dose 0.1nM

Mesma lógica da seção 6.4, agora contra `df_patent_0_1nM`. O matching por posição (seção 6.3) não
depende da dose, então reaproveitamos os mesmos `raw_*_pcsk9` da seção 6.2 — só o casamos de novo,
agora contra `df_patent_0_1nM`.

Uma diferença importante: `df_patent_0_1nM` **não tem** uma coluna de rank pronta (não existe
`rank_by_0.1nM_avg` no dataset original, só `rank_by_10nM_avg`). Por isso, pro C de "Concordância"
aqui, o rank real é derivado ordenando `efficacy_score_0.1nM_avg` (menor = mais eficaz, mesmo
critério usado no `rank_by_10nM_avg` original).

Os mesmos 4 pares avaliados: OligoFormer (batch, sem filtro), OligoFormer (batch, filtrado),
DSIR-19nt e DSIR-21nt — todos contra `df_patent_0_1nM`.


In [ ]:
def compare_with_ground_truth_01nM(matched_df, model_name):
    """Correlação (C de 'Correlação'): Spearman e Kendall entre o score do modelo e a eficácia real
    medida a 0.1nM (efficacy_score_0.1nM_avg, sinal invertido: maior = mais eficaz)."""
    valid = matched_df.dropna(subset=["model_score"]).copy()
    valid["real_effect"] = -valid["efficacy_score_0.1nM_avg"]

    if len(valid) < 3:
        print(f"[{model_name}] Poucos pares válidos ({len(valid)}) para calcular correlação.")
        return {"n": len(valid), "spearman_rho": None, "spearman_p": None, "kendall_tau": None, "kendall_p": None, "merged": valid}

    rho, p_rho = spearmanr(valid["model_score"], valid["real_effect"])
    tau, p_tau = kendalltau(valid["model_score"], valid["real_effect"])

    print(f"--- {model_name} vs eficácia real, 0.1nM (n={len(valid)}) ---")
    print(f"Spearman rho: {rho:.4f} (p={p_rho:.4g})")
    print(f"Kendall tau: {tau:.4f} (p={p_tau:.4g})")

    return {"n": len(valid), "spearman_rho": rho, "spearman_p": p_rho, "kendall_tau": tau, "kendall_p": p_tau, "merged": valid}


In [ ]:
def top_n_paired_agreement_01nM(matched_df, model_name, n_values=(5, 10, 20, 50)):
    """Concordância (C de 'Concordância'): compara o top-N do PRÓPRIO modelo (por model_score)
    contra o top-N real a 0.1nM. Como não existe rank_by_0.1nM_avg pronto, o rank real é derivado
    aqui ordenando efficacy_score_0.1nM_avg (menor = mais eficaz, mesmo critério do rank_by_10nM_avg
    original)."""
    valid = matched_df.dropna(subset=["model_score"]).copy()
    n_total = len(valid)

    valid = valid.sort_values("efficacy_score_0.1nM_avg", ascending=True)
    valid["real_rank_0_1nM"] = range(1, len(valid) + 1)

    by_model = valid.sort_values("model_score", ascending=False)
    by_real = valid.sort_values("real_rank_0_1nM", ascending=True)

    results = []
    for n in n_values:
        if n > n_total:
            print(f"[{model_name}] Top-{n}: pulado (só {n_total} siRNAs casados no total)")
            continue

        top_model = set(by_model.head(n)["Duplex Name"])
        top_real = set(by_real.head(n)["Duplex Name"])
        overlap = top_model & top_real

        chance = (n * n) / n_total
        count = len(overlap)
        print(f"[{model_name}] Top-{n}: {count}/{n} do top do modelo também está no top real "
              f"({count/n:.1%}) | esperado por acaso: ~{chance:.1f}")

        results.append({
            "n": n,
            "overlap": count,
            "fraction": count / n,
            "chance_expected": chance,
            "duplex_names": sorted(overlap),
        })

    return results


#### 6.5.1 OligoFormer (batch, sem filtro) × 0.1nM


In [ ]:
patent_vs_ol_01nM = match_patent_to_model_by_position(df_patent_0_1nM, raw_ol_pcsk9)

n_matched_ol_01nM = patent_vs_ol_01nM["matched_sirna"].notna().sum()
n_confirmed_ol_01nM = patent_vs_ol_01nM["seq_confirmed"].sum()

print(f"OligoFormer (batch, sem filtro): {n_matched_ol_01nM}/{len(df_patent_0_1nM)} siRNAs da patente casados por posição "
      f"({n_matched_ol_01nM/len(df_patent_0_1nM):.1%}), {n_confirmed_ol_01nM} confirmados por sequência "
      f"({n_confirmed_ol_01nM/max(n_matched_ol_01nM,1):.1%})")


In [ ]:
n_raw_ol_01nM = len(raw_ol_pcsk9)
n_patent_01nM = len(df_patent_0_1nM)

jaccard_ol_01nM = n_matched_ol_01nM / (n_patent_01nM + n_raw_ol_01nM - n_matched_ol_01nM)
overlap_ol_01nM = n_matched_ol_01nM / min(n_patent_01nM, n_raw_ol_01nM)
dice_ol_01nM = 2 * n_matched_ol_01nM / (n_patent_01nM + n_raw_ol_01nM)

print(f"Cobertura — patente vs OligoFormer (batch, sem filtro):")
print(f"Jaccard: {jaccard_ol_01nM:.4f} | Overlap: {overlap_ol_01nM:.4f} | Dice: {dice_ol_01nM:.4f}")


In [ ]:
val_ol_01nM = compare_with_ground_truth_01nM(patent_vs_ol_01nM, "OligoFormer (batch, sem filtro)")


In [ ]:
top_n_ol_01nM = top_n_paired_agreement_01nM(patent_vs_ol_01nM, "OligoFormer (batch, sem filtro)")


#### 6.5.2 OligoFormer (batch, filtrado) × 0.1nM


In [ ]:
patent_vs_ol_f_01nM = match_patent_to_model_by_position(df_patent_0_1nM, raw_ol_f_pcsk9)

n_matched_ol_f_01nM = patent_vs_ol_f_01nM["matched_sirna"].notna().sum()
n_confirmed_ol_f_01nM = patent_vs_ol_f_01nM["seq_confirmed"].sum()

print(f"OligoFormer (batch, filtrado): {n_matched_ol_f_01nM}/{len(df_patent_0_1nM)} siRNAs da patente casados por posição "
      f"({n_matched_ol_f_01nM/len(df_patent_0_1nM):.1%}), {n_confirmed_ol_f_01nM} confirmados por sequência "
      f"({n_confirmed_ol_f_01nM/max(n_matched_ol_f_01nM,1):.1%})")


In [ ]:
n_raw_ol_f_01nM = len(raw_ol_f_pcsk9)
n_patent_01nM = len(df_patent_0_1nM)

jaccard_ol_f_01nM = n_matched_ol_f_01nM / (n_patent_01nM + n_raw_ol_f_01nM - n_matched_ol_f_01nM)
overlap_ol_f_01nM = n_matched_ol_f_01nM / min(n_patent_01nM, n_raw_ol_f_01nM)
dice_ol_f_01nM = 2 * n_matched_ol_f_01nM / (n_patent_01nM + n_raw_ol_f_01nM)

print(f"Cobertura — patente vs OligoFormer (batch, filtrado):")
print(f"Jaccard: {jaccard_ol_f_01nM:.4f} | Overlap: {overlap_ol_f_01nM:.4f} | Dice: {dice_ol_f_01nM:.4f}")


In [ ]:
val_ol_f_01nM = compare_with_ground_truth_01nM(patent_vs_ol_f_01nM, "OligoFormer (batch, filtrado)")


In [ ]:
top_n_ol_f_01nM = top_n_paired_agreement_01nM(patent_vs_ol_f_01nM, "OligoFormer (batch, filtrado)")


#### 6.5.3 DSIR-19nt × 0.1nM


In [ ]:
patent_vs_dsir19_01nM = match_patent_to_model_by_position(df_patent_0_1nM, raw_dsir19_pcsk9)

n_matched_dsir19_01nM = patent_vs_dsir19_01nM["matched_sirna"].notna().sum()
n_confirmed_dsir19_01nM = patent_vs_dsir19_01nM["seq_confirmed"].sum()

print(f"DSIR-19nt: {n_matched_dsir19_01nM}/{len(df_patent_0_1nM)} siRNAs da patente casados por posição "
      f"({n_matched_dsir19_01nM/len(df_patent_0_1nM):.1%}), {n_confirmed_dsir19_01nM} confirmados por sequência "
      f"({n_confirmed_dsir19_01nM/max(n_matched_dsir19_01nM,1):.1%})")


In [ ]:
n_raw_dsir19_01nM = len(raw_dsir19_pcsk9)
n_patent_01nM = len(df_patent_0_1nM)

jaccard_dsir19_01nM = n_matched_dsir19_01nM / (n_patent_01nM + n_raw_dsir19_01nM - n_matched_dsir19_01nM)
overlap_dsir19_01nM = n_matched_dsir19_01nM / min(n_patent_01nM, n_raw_dsir19_01nM)
dice_dsir19_01nM = 2 * n_matched_dsir19_01nM / (n_patent_01nM + n_raw_dsir19_01nM)

print(f"Cobertura — patente vs DSIR-19nt:")
print(f"Jaccard: {jaccard_dsir19_01nM:.4f} | Overlap: {overlap_dsir19_01nM:.4f} | Dice: {dice_dsir19_01nM:.4f}")


In [ ]:
val_dsir19_01nM = compare_with_ground_truth_01nM(patent_vs_dsir19_01nM, "DSIR-19nt")


In [ ]:
top_n_dsir19_01nM = top_n_paired_agreement_01nM(patent_vs_dsir19_01nM, "DSIR-19nt")


#### 6.5.4 DSIR-21nt × 0.1nM


In [ ]:
patent_vs_dsir21_01nM = match_patent_to_model_by_position(df_patent_0_1nM, raw_dsir21_pcsk9)

n_matched_dsir21_01nM = patent_vs_dsir21_01nM["matched_sirna"].notna().sum()
n_confirmed_dsir21_01nM = patent_vs_dsir21_01nM["seq_confirmed"].sum()

print(f"DSIR-21nt: {n_matched_dsir21_01nM}/{len(df_patent_0_1nM)} siRNAs da patente casados por posição "
      f"({n_matched_dsir21_01nM/len(df_patent_0_1nM):.1%}), {n_confirmed_dsir21_01nM} confirmados por sequência "
      f"({n_confirmed_dsir21_01nM/max(n_matched_dsir21_01nM,1):.1%})")


In [ ]:
n_raw_dsir21_01nM = len(raw_dsir21_pcsk9)
n_patent_01nM = len(df_patent_0_1nM)

jaccard_dsir21_01nM = n_matched_dsir21_01nM / (n_patent_01nM + n_raw_dsir21_01nM - n_matched_dsir21_01nM)
overlap_dsir21_01nM = n_matched_dsir21_01nM / min(n_patent_01nM, n_raw_dsir21_01nM)
dice_dsir21_01nM = 2 * n_matched_dsir21_01nM / (n_patent_01nM + n_raw_dsir21_01nM)

print(f"Cobertura — patente vs DSIR-21nt:")
print(f"Jaccard: {jaccard_dsir21_01nM:.4f} | Overlap: {overlap_dsir21_01nM:.4f} | Dice: {dice_dsir21_01nM:.4f}")


In [ ]:
val_dsir21_01nM = compare_with_ground_truth_01nM(patent_vs_dsir21_01nM, "DSIR-21nt")


In [ ]:
top_n_dsir21_01nM = top_n_paired_agreement_01nM(patent_vs_dsir21_01nM, "DSIR-21nt")


---

## 7. Comparações e checagens finais

Com os dataframes e resultados de validação já prontos (seções 4–6), essa seção reúne checagens e comparações adicionais sobre os dados que já temos, com seus respectivos gráficos.

### 7.1 Não-determinismo do OligoFormer (batch de 10 runs)

O OligoFormer não é determinístico — o mesmo fasta pode gerar scores um pouco diferentes a cada execução. `df_ol_batch` (seção 4.3) já guarda, por siRNA, a média, o desvio-padrão e o min/max da eficácia entre as 10 runs do batch; as duas células abaixo só olham para essa variação mais de perto, pra caracterizar o quão estável (ou instável) o modelo é run a run.

In [ ]:
print(df_ol_batch["efficacy_std"].describe())

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(df_ol_batch["efficacy_std"], bins=40)
ax.set_xlabel("Desvio-padrão da eficácia entre os 10 runs (por siRNA)")
ax.set_ylabel("Número de siRNAs")
ax.set_title("Não-determinismo do OligoFormer: variação de score entre runs (PCSK9)")
plt.tight_layout()
plt.show()


In [ ]:
N_SIRNAS_PLOT = 40  # amostra para não poluir o eixo x; ajuste conforme necessário

df_plot = df_ol_batch.sort_values("efficacy_mean", ascending=False).head(N_SIRNAS_PLOT).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(df_plot))

ax.vlines(x, df_plot["efficacy_min"], df_plot["efficacy_max"], color="gray", alpha=0.5, linewidth=1.5,
          label="Min–Max (10 runs)")
ax.errorbar(x, df_plot["efficacy_mean"], yerr=df_plot["efficacy_std"], fmt="o", color="#4C72B0",
            markersize=4, capsize=2, label="Média ± desvio-padrão")

ax.set_xticks(x)
ax.set_xticklabels(df_plot["sirna"], rotation=90, fontsize=6)
ax.set_ylabel("Eficácia predita (OligoFormer)")
ax.set_title(f"Não-determinismo do OligoFormer — top {N_SIRNAS_PLOT} siRNAs por eficácia média (PCSK9)")
ax.legend()
plt.tight_layout()
plt.show()


### 7.2 Não-determinismo do OligoFormer filtrado (batch de 10 runs)

Mesma análise da seção 7.1, agora para `df_ol_batch_filtered` (seção 4.4) — os candidatos que passam pelo filtro interno do OligoFormer (`filters["filter"] == 0"`).

In [ ]:
print(df_ol_batch_filtered["efficacy_std"].describe())

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(df_ol_batch_filtered["efficacy_std"], bins=40)
ax.set_xlabel("Desvio-padrão da eficácia entre os 10 runs (por siRNA)")
ax.set_ylabel("Número de siRNAs")
ax.set_title("Não-determinismo do OligoFormer filtrado: variação de score entre runs (PCSK9)")
plt.tight_layout()
plt.show()


In [ ]:
N_SIRNAS_PLOT = 40  # amostra para não poluir o eixo x; ajuste conforme necessário

df_plot_f = df_ol_batch_filtered.sort_values("efficacy_mean", ascending=False).head(N_SIRNAS_PLOT).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(df_plot_f))

ax.vlines(x, df_plot_f["efficacy_min"], df_plot_f["efficacy_max"], color="gray", alpha=0.5, linewidth=1.5,
          label="Min–Max (10 runs)")
ax.errorbar(x, df_plot_f["efficacy_mean"], yerr=df_plot_f["efficacy_std"], fmt="o", color="#DD8452",
            markersize=4, capsize=2, label="Média ± desvio-padrão")

ax.set_xticks(x)
ax.set_xticklabels(df_plot_f["sirna"], rotation=90, fontsize=6)
ax.set_ylabel("Eficácia predita (OligoFormer filtrado)")
ax.set_title(f"Não-determinismo do OligoFormer filtrado — top {N_SIRNAS_PLOT} siRNAs por eficácia média (PCSK9)")
ax.legend()
plt.tight_layout()
plt.show()


### 7.3 OligoFormer vs OligoFormer filtrado: o filtro reduz a variação entre runs?

Comparação direta do desvio-padrão da eficácia entre as 10 runs (`efficacy_std`), sem filtro (seção 7.1) vs filtrado (seção 7.2), para os siRNAs em comum aos dois conjuntos.

In [ ]:
common_sirnas_var = set(df_ol_batch["sirna"]) & set(df_ol_batch_filtered["sirna"])

std_unf = df_ol_batch.set_index("sirna").loc[list(common_sirnas_var), "efficacy_std"]
std_f = df_ol_batch_filtered.set_index("sirna").loc[list(common_sirnas_var), "efficacy_std"]

print(f"siRNAs em comum (sem filtro ∩ filtrado): {len(common_sirnas_var)}")
print(f"Desvio-padrão médio — sem filtro : {std_unf.mean():.4f}")
print(f"Desvio-padrão médio — filtrado   : {std_f.mean():.4f}")

fig, ax = plt.subplots(figsize=(7, 4.5))
bins = np.linspace(0, max(std_unf.max(), std_f.max()), 40)
ax.hist(std_unf, bins=bins, alpha=0.6, label=f"Sem filtro (média={std_unf.mean():.3f})", color="#4C72B0")
ax.hist(std_f, bins=bins, alpha=0.6, label=f"Filtrado (média={std_f.mean():.3f})", color="#DD8452")
ax.set_xlabel("Desvio-padrão da eficácia entre os 10 runs (por siRNA)")
ax.set_ylabel("Número de siRNAs")
ax.set_title("Variação entre runs: OligoFormer sem filtro vs filtrado (siRNAs em comum, PCSK9)")
ax.legend()
plt.tight_layout()
plt.show()


### 7.4 Juntando os 3 C's: tabela-resumo por modelo (10nM e 0.1nM)

As seções 6.4 e 6.5 calcularam cobertura (Jaccard/Overlap/Dice), correlação (Spearman/Kendall) e
concordância (top-N pareado) para os 4 modelos/variantes, mas cada métrica ficou numa variável
separada. Aqui juntamos tudo numa única tabela por dose, resumindo a concordância como a média das
frações de top-N pareado (N=5,10,20,50).

In [ ]:
def build_c3_summary(models):
    """Junta cobertura (Jaccard/Overlap/Dice), correlação (Spearman/Kendall) e concordância
    (média das frações de top-N pareado) numa única tabela, uma linha por modelo/variante.

    models: dict {nome: (jaccard, overlap, dice, val_dict, top_n_list)}
    """
    rows = []
    for nome, (j, o, d, val, topn) in models.items():
        fracs = [r["fraction"] for r in topn]
        rows.append({
            "modelo": nome,
            "cobertura_jaccard": j,
            "cobertura_overlap": o,
            "cobertura_dice": d,
            "correlacao_spearman": val["spearman_rho"],
            "correlacao_kendall": val["kendall_tau"],
            "concordancia_topN_media": np.mean(fracs) if fracs else np.nan,
        })
    return pd.DataFrame(rows).set_index("modelo")


models_10nM = {
    "OligoFormer (sem filtro)": (jaccard_ol_10nM, overlap_ol_10nM, dice_ol_10nM, val_ol_10nM, top_n_ol_10nM),
    "OligoFormer (filtrado)":   (jaccard_ol_f_10nM, overlap_ol_f_10nM, dice_ol_f_10nM, val_ol_f_10nM, top_n_ol_f_10nM),
    "DSIR-19nt":                (jaccard_dsir19_10nM, overlap_dsir19_10nM, dice_dsir19_10nM, val_dsir19_10nM, top_n_dsir19_10nM),
    "DSIR-21nt":                (jaccard_dsir21_10nM, overlap_dsir21_10nM, dice_dsir21_10nM, val_dsir21_10nM, top_n_dsir21_10nM),
}

models_01nM = {
    "OligoFormer (sem filtro)": (jaccard_ol_01nM, overlap_ol_01nM, dice_ol_01nM, val_ol_01nM, top_n_ol_01nM),
    "OligoFormer (filtrado)":   (jaccard_ol_f_01nM, overlap_ol_f_01nM, dice_ol_f_01nM, val_ol_f_01nM, top_n_ol_f_01nM),
    "DSIR-19nt":                (jaccard_dsir19_01nM, overlap_dsir19_01nM, dice_dsir19_01nM, val_dsir19_01nM, top_n_dsir19_01nM),
    "DSIR-21nt":                (jaccard_dsir21_01nM, overlap_dsir21_01nM, dice_dsir21_01nM, val_dsir21_01nM, top_n_dsir21_01nM),
}

c3_10nM = build_c3_summary(models_10nM)
c3_01nM = build_c3_summary(models_01nM)

print("=== Resumo dos 3 C's — dose 10nM ===")
display(c3_10nM.round(4))
print()
print("=== Resumo dos 3 C's — dose 0.1nM ===")
display(c3_01nM.round(4))


### 7.5 Cobertura: Jaccard / Overlap / Dice, os 4 modelos lado a lado

Mesmo estilo do gráfico de barras agrupadas usado na seção do MAPT ("Concordância com o DSIR:
OligoFormer filtrado vs sem filtro"), agora com os 4 modelos/variantes lado a lado (cores
diferentes) para cada métrica de cobertura, um painel por dose.

In [ ]:
import os
os.makedirs("final_results", exist_ok=True)

cores_modelos = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]  # mesma ordem/cores em todos os 3 C's


def bar_chart_by_c(ax, df, metric_cols, metric_labels, cores, titulo):
    """Gráfico de barras agrupadas: uma coluna por métrica (metric_cols), com uma sub-barra por
    modelo (linhas de df), replicando o padrão usado no gráfico da seção MAPT."""
    modelos = df.index.tolist()
    x = np.arange(len(metric_cols))
    largura = 0.8 / len(modelos)

    for i, modelo in enumerate(modelos):
        valores = df.loc[modelo, metric_cols].astype(float).values
        offset = (i - (len(modelos) - 1) / 2) * largura
        ax.bar(x + offset, valores, largura, label=modelo, color=cores[i])

    ax.set_xticks(x)
    ax.set_xticklabels(metric_labels)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(titulo, fontsize=10)


metric_cols_cobertura = ["cobertura_jaccard", "cobertura_overlap", "cobertura_dice"]
metric_labels_cobertura = ["Jaccard", "Overlap", "Dice"]

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
for ax, dose_label, df in zip(axes, ["10nM", "0.1nM"], [c3_10nM, c3_01nM]):
    bar_chart_by_c(ax, df, metric_cols_cobertura, metric_labels_cobertura, cores_modelos, dose_label)

axes[0].set_ylabel("Valor da métrica")
axes[0].legend(fontsize=8, loc="upper right")
plt.suptitle("Cobertura — patente vs modelo (Jaccard / Overlap / Dice)", y=1.03)
plt.tight_layout()
plt.savefig("final_results/fig_c3_cobertura.png", bbox_inches="tight")
plt.show()


### 7.6 Correlação: Spearman rho / Kendall tau, os 4 modelos lado a lado

Mesma estrutura da 7.5, agora para as métricas de correlação com a eficácia real.

In [ ]:
import os
os.makedirs("final_results", exist_ok=True)

metric_cols_correlacao = ["correlacao_spearman", "correlacao_kendall"]
metric_labels_correlacao = ["Spearman rho", "Kendall tau"]

fig, axes = plt.subplots(1, 2, figsize=(10, 5), sharey=True)
for ax, dose_label, df in zip(axes, ["10nM", "0.1nM"], [c3_10nM, c3_01nM]):
    bar_chart_by_c(ax, df, metric_cols_correlacao, metric_labels_correlacao, cores_modelos, dose_label)

axes[0].set_ylabel("Valor da métrica")
axes[0].legend(fontsize=8, loc="upper right")
plt.suptitle("Correlação com a eficácia real (Spearman / Kendall)", y=1.03)
plt.tight_layout()
plt.savefig("final_results/fig_c3_correlacao.png", bbox_inches="tight")
plt.show()


### 7.7 Concordância: top-N pareado, os 4 modelos lado a lado

Mesma estrutura da 7.5/7.6, mas aqui cada coluna é um valor de N (5, 10, 20, 50) em vez de uma
métrica separada — usa as frações de `top_n_paired_agreement` direto (a 7.4 só guardava a média
delas, aqui usamos cada N individualmente).

In [ ]:
import os
os.makedirs("final_results", exist_ok=True)

def build_topn_matrix(topn_por_modelo):
    """topn_por_modelo: dict {modelo: top_n_list} -> DataFrame (modelo x Top-N), valores = fraction."""
    ns = sorted({r["n"] for lst in topn_por_modelo.values() for r in lst})
    colunas = [f"Top-{n}" for n in ns]
    df = pd.DataFrame(index=list(topn_por_modelo.keys()), columns=colunas, dtype=float)
    for modelo, lst in topn_por_modelo.items():
        for r in lst:
            df.loc[modelo, f"Top-{r['n']}"] = r["fraction"]
    return df


topn_10nM = {
    "OligoFormer (sem filtro)": top_n_ol_10nM,
    "OligoFormer (filtrado)":   top_n_ol_f_10nM,
    "DSIR-19nt":                top_n_dsir19_10nM,
    "DSIR-21nt":                top_n_dsir21_10nM,
}
topn_01nM = {
    "OligoFormer (sem filtro)": top_n_ol_01nM,
    "OligoFormer (filtrado)":   top_n_ol_f_01nM,
    "DSIR-19nt":                top_n_dsir19_01nM,
    "DSIR-21nt":                top_n_dsir21_01nM,
}

topn_df_10nM = build_topn_matrix(topn_10nM)
topn_df_01nM = build_topn_matrix(topn_01nM)
metric_cols_concordancia = topn_df_10nM.columns.tolist()

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
for ax, dose_label, df in zip(axes, ["10nM", "0.1nM"], [topn_df_10nM, topn_df_01nM]):
    bar_chart_by_c(ax, df, metric_cols_concordancia, metric_cols_concordancia, cores_modelos, dose_label)

axes[0].set_ylabel("Fração em comum com o top-N real")
axes[0].legend(fontsize=8, loc="upper right")
plt.suptitle("Concordância — top-N pareado (patente vs modelo)", y=1.03)
plt.tight_layout()
plt.savefig("final_results/fig_c3_concordancia.png", bbox_inches="tight")
plt.show()


### 7.8 Correlação de rank entre DSIR e OligoFormer (scatter)

Retomando o gráfico antigo (seção 12.4), que estava quebrado (`corr_pcsk9` nunca foi definido —
o nome atual é `corr_ol_batch_pcsk9`) e tinha dois problemas estatísticos:

**(a) Reta de regressão linear (OLS) sobre uma estatística rank-based.** O gráfico antigo plotava
os scores brutos com uma reta `np.polyfit` (mínimos quadrados), mas o número reportado no título é
Spearman rho — que mede correlação de **rank**, não linearidade dos valores brutos. Para não plotar
uma coisa e reportar outra, aqui: (1) os eixos passam a ser os **ranks** de cada score (rank 1 =
maior score), o que é literalmente o que o Spearman rho mede; e (2) a reta ajustada é o
**estimador de Theil-Sen** (mediana das inclinações par-a-par) em vez de OLS — é um método robusto
e não-paramétrico, consistente com o espírito rank-based de Spearman/Kendall, e não é puxado por
outliers do jeito que OLS é.

**(b) A correlação é condicionada à interseção dos conjuntos.** `n_common` (n na legenda de cada
painel) é o número de siRNAs que aparecem **nos dois** conjuntos de candidatos — não o total de
candidatos de cada modelo. Isso é deixado explícito no subtítulo da figura, para não ser lido como
"correlação geral entre os dois modelos" quando na verdade é "correlação entre os candidatos que
ambos os modelos escolheram considerar".

In [ ]:
import os
os.makedirs("final_results", exist_ok=True)

from scipy.stats import rankdata, theilslopes


def plot_rank_scatter(ax, corr_, titulo):
    """Scatter de rank(DSIR) vs rank(OligoFormer), com ajuste Theil-Sen (robusto, rank-based)
    no lugar de uma reta OLS — consistente com o Spearman/Kendall reportados no título."""
    m = corr_["merged"]
    rank_dsir = rankdata(-m["dsir"])  # rank 1 = maior score
    rank_ol = rankdata(-m["ol"])

    ax.scatter(rank_dsir, rank_ol, alpha=0.6, s=20)

    slope, intercept, _lo, _hi = theilslopes(rank_ol, rank_dsir)
    xs = np.linspace(rank_dsir.min(), rank_dsir.max(), 50)
    ax.plot(xs, slope * xs + intercept, color="red", linestyle="--", linewidth=1.5,
            label="Ajuste Theil-Sen (robusto, rank-based)")

    ax.set_xlabel("Rank DSIR (1 = maior score)")
    ax.set_ylabel("Rank OligoFormer (1 = maior score)")
    ax.set_title(
        f"{titulo} — Spearman ρ = {corr_['spearman_rho']:.3f}, "
        f"Kendall τ = {corr_['kendall_tau']:.3f} (n={corr_['n_common']})",
        fontsize=10,
    )
    ax.legend(fontsize=7, loc="upper left")


fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
plot_rank_scatter(axes[0], corr, "MAPT")
plot_rank_scatter(axes[1], corr_ol_batch_pcsk9, "PCSK9 (OligoFormer batch, sem filtro)")

plt.suptitle(
    "Correlação de rank entre DSIR e OligoFormer\n"
    "Amostra: apenas siRNAs na interseção dos dois conjuntos — não representa o total de candidatos de cada modelo",
    y=1.08, fontsize=10,
)
plt.tight_layout()
plt.savefig("final_results/fig_rank_correlation_scatter.png", bbox_inches="tight")
plt.show()


### 7.9 Scatter score do modelo vs eficácia real (validação com a patente)

Retomando o gráfico antigo (seção 12.5), que também estava quebrado (`val_dsir`, `val_dsir21`,
`val_ol`, `val_ol_specific` nunca existiram nesse formato — os nomes atuais têm sufixo de dose, ex.
`val_dsir19_10nM`) e tinha quatro problemas estatísticos identificados antes. Correções aplicadas:

**Tamanho amostral pequeno → intervalo de confiança via bootstrap.** Em vez de reportar só o
Spearman rho pontual, cada painel mostra um IC 95% (bootstrap percentil, 1000 reamostragens dos
pares modelo-vs-real com reposição). `n` (o tamanho da amostra pareada com a patente) aparece
destacado no título de cada painel — é ele que determina o quão largo o IC fica.

**Comparações múltiplas sem correção → checagem visual de sobreposição de IC's.** Com 4
modelos/variantes × 2 doses = 8 correlações, uma célula à parte lista, por dose, quais pares de
modelos têm IC's sobrepostos — quando sobrepõem, a diferença observada entre eles não deve ser lida
como uma diferença real de desempenho.

**Reta OLS sobre estatística rank-based → mesma correção da 7.8.** Os eixos passam a ser os
**ranks** de `model_score` e `real_effect`, e a reta ajustada é o estimador de **Theil-Sen**
(robusto, não-paramétrico) em vez de mínimos quadrados.

**Dependência do matching por posição → checagem de sensibilidade.** Uma célula recalcula o
matching e a correlação com `position_tolerance=1` (em vez do `0` usado na seção 6.3) para os 4
modelos × 2 doses, e compara `n` e rho antes/depois — se os números mudarem muito, o resultado
principal depende fortemente dessa escolha de tolerância.

In [ ]:
from scipy.stats import spearmanr


def bootstrap_spearman_ci(merged_df, x_col="model_score", y_col="real_effect", n_boot=1000, ci=95, random_state=42):
    """IC bootstrap (percentil) para o Spearman rho, reamostrando pares (x, y) com reposição."""
    rng = np.random.default_rng(random_state)
    x = merged_df[x_col].to_numpy()
    y = merged_df[y_col].to_numpy()
    n = len(x)
    if n < 3:
        return np.nan, np.nan

    rhos = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        rhos[b] = spearmanr(x[idx], y[idx])[0]

    lo, hi = np.percentile(rhos, [(100 - ci) / 2, 100 - (100 - ci) / 2])
    return lo, hi


modelos_val = {
    "OligoFormer (sem filtro)": (val_ol_10nM, val_ol_01nM),
    "OligoFormer (filtrado)":   (val_ol_f_10nM, val_ol_f_01nM),
    "DSIR-19nt":                (val_dsir19_10nM, val_dsir19_01nM),
    "DSIR-21nt":                (val_dsir21_10nM, val_dsir21_01nM),
}

linhas_ci = []
for modelo_nome, (val_10, val_01) in modelos_val.items():
    for dose_label, val in [("10nM", val_10), ("0.1nM", val_01)]:
        ci_lo, ci_hi = bootstrap_spearman_ci(val["merged"])
        linhas_ci.append({
            "modelo": modelo_nome,
            "dose": dose_label,
            "n": val["n"],
            "spearman_rho": val["spearman_rho"],
            "ci95_lo": ci_lo,
            "ci95_hi": ci_hi,
        })

df_ci_rho = pd.DataFrame(linhas_ci)
print("=== Spearman rho com IC 95% (bootstrap, 1000 reamostragens) ===")
display(df_ci_rho.round(4))


In [ ]:
print("=== Atenção a comparações múltiplas: pares de modelos com IC's sobrepostos (mesma dose) ===")
for dose_label in ["10nM", "0.1nM"]:
    sub = df_ci_rho[df_ci_rho["dose"] == dose_label].reset_index(drop=True)
    print(f"--- {dose_label} ---")
    for i in range(len(sub)):
        for j in range(i + 1, len(sub)):
            a, b = sub.loc[i], sub.loc[j]
            if pd.isna(a["ci95_lo"]) or pd.isna(b["ci95_lo"]):
                continue
            overlap = not (a["ci95_hi"] < b["ci95_lo"] or b["ci95_hi"] < a["ci95_lo"])
            status = "IC's SE SOBREPÕEM -> diferença pode não ser real" if overlap else "IC's não se sobrepõem"
            print(f"{a['modelo']} vs {b['modelo']}: {status}")
    print()


In [ ]:
sensibilidade_tolerancia = []

modelos_raw = {
    "OligoFormer (sem filtro)": raw_ol_pcsk9,
    "OligoFormer (filtrado)":   raw_ol_f_pcsk9,
    "DSIR-19nt":                raw_dsir19_pcsk9,
    "DSIR-21nt":                raw_dsir21_pcsk9,
}

doses_cfg = [
    ("10nM", df_patent_10nM, compare_with_ground_truth),
    ("0.1nM", df_patent_0_1nM, compare_with_ground_truth_01nM),
]

for dose_label, df_patent_dose, compare_fn in doses_cfg:
    for modelo_nome, raw in modelos_raw.items():
        m_tol0 = match_patent_to_model_by_position(df_patent_dose, raw, position_tolerance=0)
        m_tol1 = match_patent_to_model_by_position(df_patent_dose, raw, position_tolerance=1)

        n0 = int(m_tol0["matched_sirna"].notna().sum())
        n1 = int(m_tol1["matched_sirna"].notna().sum())

        val0 = compare_fn(m_tol0, f"{modelo_nome} ({dose_label}, tol=0)")
        val1 = compare_fn(m_tol1, f"{modelo_nome} ({dose_label}, tol=1)")

        sensibilidade_tolerancia.append({
            "dose": dose_label,
            "modelo": modelo_nome,
            "n_tol0": n0, "rho_tol0": val0["spearman_rho"],
            "n_tol1": n1, "rho_tol1": val1["spearman_rho"],
        })

df_sensibilidade = pd.DataFrame(sensibilidade_tolerancia)
df_sensibilidade["delta_n"] = df_sensibilidade["n_tol1"] - df_sensibilidade["n_tol0"]
df_sensibilidade["delta_rho"] = df_sensibilidade["rho_tol1"] - df_sensibilidade["rho_tol0"]

print()
print("=== Sensibilidade do matching por posição: tolerância 0 vs 1 ===")
display(df_sensibilidade[["dose", "modelo", "n_tol0", "n_tol1", "delta_n", "rho_tol0", "rho_tol1", "delta_rho"]].round(4))


In [ ]:
import os
os.makedirs("final_results", exist_ok=True)

from scipy.stats import rankdata, theilslopes


def plot_model_vs_real(ax, val, ci_row, titulo, cor):
    m = val["merged"]
    if len(m) < 3:
        ax.set_title(f"{titulo}\n(n={val['n']}, pares insuficientes)", fontsize=8)
        ax.axis("off")
        return

    rank_model = rankdata(-m["model_score"])
    rank_real = rankdata(-m["real_effect"])

    ax.scatter(rank_model, rank_real, alpha=0.6, s=18, color=cor)

    slope, intercept, _lo, _hi = theilslopes(rank_real, rank_model)
    xs = np.linspace(rank_model.min(), rank_model.max(), 50)
    ax.plot(xs, slope * xs + intercept, color="black", linestyle="--", linewidth=1.2)

    ax.set_xlabel("Rank score modelo", fontsize=8)
    ax.set_ylabel("Rank eficácia real", fontsize=8)
    ax.set_title(
        f"{titulo}\nρ={ci_row['spearman_rho']:.3f} [{ci_row['ci95_lo']:.2f}, {ci_row['ci95_hi']:.2f}] · n={ci_row['n']}",
        fontsize=8,
    )


fig, axes = plt.subplots(2, 4, figsize=(16, 7.5))

for col, (modelo_nome, (val_10, val_01)) in enumerate(modelos_val.items()):
    ci_10 = df_ci_rho[(df_ci_rho["modelo"] == modelo_nome) & (df_ci_rho["dose"] == "10nM")].iloc[0]
    ci_01 = df_ci_rho[(df_ci_rho["modelo"] == modelo_nome) & (df_ci_rho["dose"] == "0.1nM")].iloc[0]
    plot_model_vs_real(axes[0, col], val_10, ci_10, f"{modelo_nome} — 10nM", cores_modelos[col])
    plot_model_vs_real(axes[1, col], val_01, ci_01, f"{modelo_nome} — 0.1nM", cores_modelos[col])

plt.suptitle(
    "Score do modelo vs eficácia real medida (validação PCSK9)\n"
    "Eixos em rank · reta = ajuste Theil-Sen · ρ com IC 95% via bootstrap",
    y=1.03, fontsize=10,
)
plt.tight_layout()
plt.savefig("final_results/fig_model_vs_real_efficacy.png", bbox_inches="tight")
plt.show()


### 7.10 Variação do OligoFormer entre execuções (batch de 10 runs)

Retomando o gráfico antigo (seção 12.7) — esse era o mais seguro dos quatro (`df_ol_batch` continua
válido, sem variável quebrada, e é só estatística descritiva, sem teste de hipótese). Os dois
cuidados do veredito, ambos sobre confiabilidade das estimativas com apenas n=10 execuções:

**Min–max com n=10 subestima a amplitude real.** O range observado em só 10 amostras é, em
expectativa, menor que o range verdadeiro da distribuição subjacente. Assumindo aproximadamente
normalidade entre as execuções, a constante de Shewhart d2(10) ≈ 3.078 dá a relação clássica
E[range] ≈ d2 · σ — usamos isso para comparar, em texto, o desvio-padrão amostral com o σ implícito
no range observado, e deixamos a ressalva explícita no título/legenda do gráfico.

**O próprio desvio-padrão de n=10 é incerto.** Adicionamos um IC 95% para o desvio-padrão
populacional de cada siRNA via a distribuição qui-quadrado (Fisher: (n-1)·s²/σ² ~ χ²₍ₙ₋₁₎),
mostrado como uma faixa larga e translúcida atrás da barra de erro-padrão já existente — o quanto
essa faixa "abre" mostra o quanto o `efficacy_std` reportado pode estar longe do desvio-padrão
populacional real.

In [ ]:
from scipy.stats import chi2

D2_N10 = 3.078  # constante de Shewhart (d2) para n=10: E[range] ≈ d2 · sigma, sob normalidade


def std_ci_chi2(s, n, ci=95):
    """IC para o desvio-padrão populacional a partir do desvio-padrão amostral s (n observações),
    assumindo runs aproximadamente normais: (n-1)*s^2/sigma^2 ~ chi2_(n-1)."""
    if pd.isna(s) or s == 0:
        return 0.0, 0.0
    alpha = 1 - ci / 100
    df = n - 1
    chi2_lo = chi2.ppf(alpha / 2, df)
    chi2_hi = chi2.ppf(1 - alpha / 2, df)
    sigma_lo = s * np.sqrt(df / chi2_hi)
    sigma_hi = s * np.sqrt(df / chi2_lo)
    return sigma_lo, sigma_hi


N_SIRNAS_PLOT = 40
N_RUNS = N_BATCH_RUNS  # 10, definido na seção 4.3

df_plot = df_ol_batch.sort_values("efficacy_mean", ascending=False).head(N_SIRNAS_PLOT).reset_index(drop=True)

ci_bounds = df_plot["efficacy_std"].apply(lambda s: std_ci_chi2(s, N_RUNS))
df_plot["std_ci_lo"] = [b[0] for b in ci_bounds]
df_plot["std_ci_hi"] = [b[1] for b in ci_bounds]

sigma_estimado_pelo_range = (df_plot["efficacy_max"] - df_plot["efficacy_min"]) / D2_N10
razao_media = (df_plot["efficacy_std"] / sigma_estimado_pelo_range).replace([np.inf, -np.inf], np.nan).mean()

print(f"Constante de Shewhart d2({N_RUNS}) = {D2_N10}  (E[range] ≈ d2 · σ, sob normalidade)")
print(f"Razão média efficacy_std / (range / d2) entre os {N_SIRNAS_PLOT} siRNAs plotados: {razao_media:.3f}")
print("Se essa razão for << 1 ou >> 1, as duas estimativas de variação (desvio-padrão amostral vs "
      "range corrigido) discordam — de qualquer forma, ambas vêm de só n=10 runs, daí o IC abaixo.")


In [ ]:
import os
os.makedirs("final_results", exist_ok=True)

fig, ax = plt.subplots(figsize=(12, 5.5))
x = np.arange(len(df_plot))

# IC 95% do desvio-padrão populacional (faixa larga e translúcida, atrás de tudo)
ax.errorbar(x, df_plot["efficacy_mean"], yerr=df_plot["std_ci_hi"], fmt="none",
            ecolor="#4C72B0", alpha=0.15, elinewidth=6, capsize=0, zorder=1,
            label="IC 95% do desvio-padrão populacional (χ², n=10)")

# Min-Max observado (10 runs)
ax.vlines(x, df_plot["efficacy_min"], df_plot["efficacy_max"], color="gray", alpha=0.5, linewidth=1.5,
          zorder=2, label="Min–Max observado (10 runs) — tende a subestimar a variação real")

# Média ± desvio-padrão amostral
ax.errorbar(x, df_plot["efficacy_mean"], yerr=df_plot["efficacy_std"], fmt="o", color="#4C72B0",
            markersize=4, capsize=2, zorder=3, label="Média ± desvio-padrão amostral")

ax.set_xticks(x)
ax.set_xticklabels(df_plot["sirna"], rotation=90, fontsize=6)
ax.set_ylabel("Eficácia predita (OligoFormer)")
ax.set_title(
    f"Não-determinismo do OligoFormer — top {N_SIRNAS_PLOT} siRNAs por eficácia média (PCSK9, n={N_RUNS} runs)\n"
    "Min–Max e desvio-padrão estimados com apenas 10 execuções — ambos tendem a subestimar a variação real",
    fontsize=10,
)
ax.legend(fontsize=7, loc="upper right")
plt.tight_layout()
plt.savefig("final_results/fig_oligoformer_variation.png", bbox_inches="tight")
plt.show()


### 7.11 Matrizes de confusão: classe "efetivo" / "não efetivo"

Define classes binárias a partir de thresholds fixos (não das métricas de rank usadas até aqui):

- **DSIR** (19nt e 21nt): score ≥ **0.9** = efetivo (default do paper do DSIR).
- **OligoFormer** (sem filtro e filtrado): score ≥ **0.7** = efetivo (o paper do OligoFormer não
  define um default; 0.7 é a escolha adotada aqui).
- **Eficácia real (patente)**: confirmado no texto da patente (WO2014089313A1, Tabela 4) que
  `efficacy_score_10nM_avg` é a fração de mRNA remanescente em relação ao controle (escala 0-1,
  inibição = 1 − remanescente). Para inibição > 70%: mRNA remanescente ≤ 0.30 = efetivo.

**Assumindo a dose 10nM** (não especificada no pedido) — troque `_10nM` por `_01nM` nos dicts
abaixo se quiser a 0.1nM em vez disso.

5 matrizes: OligoFormer (sem filtro), OligoFormer (filtrado), DSIR-19nt e DSIR-21nt — cada um
contra a classe real —, e por último OligoFormer (filtrado) vs DSIR-19nt (modelo contra modelo).

In [ ]:
import os
os.makedirs("final_results", exist_ok=True)

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, f1_score

LIMIAR_DSIR = 0.9  # default do paper do DSIR
LIMIAR_OL = 0.7    # paper do OligoFormer não define um default; escolha própria

# efficacy_score_10nM_avg = fração de mRNA remanescente vs controle (confirmado na patente,
# WO2014089313A1, Tabela 4: 'fraction of the message remaining relative to control').
# Inibição = 1 - remanescente -> inibição > 70% equivale a remanescente <= 0.30.
LIMIAR_REMANESCENTE = 0.30
print(f"Limiar de mRNA remanescente = {LIMIAR_REMANESCENTE} (equivalente a >70% de inibição, "
      f"fração 0-1 conforme definição da patente)")


def classes_binarias(merged_df, limiar_modelo):
    """(classe_real, classe_modelo) binárias (1 = efetivo) a partir de uma tabela merged de
    val_*_10nM (já casada com a patente)."""
    classe_real = (merged_df["efficacy_score_10nM_avg"] <= LIMIAR_REMANESCENTE).astype(int)
    classe_modelo = (merged_df["model_score"] >= limiar_modelo).astype(int)
    return classe_real, classe_modelo


def plot_confusion(ax, y_true, y_pred, titulo):
    cm = confusion_matrix(y_true, y_pred, labels=[1, 0])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Efetivo", "Não efetivo"])
    disp.plot(ax=ax, colorbar=False, cmap="Blues", values_format="d")
    ax.set_title(titulo, fontsize=10)


modelos_vs_real = [
    ("OligoFormer (sem filtro)", val_ol_10nM["merged"], LIMIAR_OL),
    ("OligoFormer (filtrado)", val_ol_f_10nM["merged"], LIMIAR_OL),
    ("DSIR-19nt", val_dsir19_10nM["merged"], LIMIAR_DSIR),
    ("DSIR-21nt", val_dsir21_10nM["merged"], LIMIAR_DSIR),
]

classificacao_binaria_10nM = {
    "limiares": {
        "dsir": LIMIAR_DSIR,
        "oligoformer": LIMIAR_OL,
        "mrna_remanescente_max_efetivo": LIMIAR_REMANESCENTE,
    },
}

fig, axes = plt.subplots(2, 2, figsize=(10, 9))
for ax, (nome, merged, limiar) in zip(axes.flat, modelos_vs_real):
    classe_real, classe_modelo = classes_binarias(merged, limiar)
    cm = confusion_matrix(classe_real, classe_modelo, labels=[1, 0])
    plot_confusion(ax, classe_real, classe_modelo, f"{nome}\n(n={len(merged)})")
    classificacao_binaria_10nM[nome] = {"n": len(merged), "confusion_matrix": cm}

plt.suptitle("Validação: classe predita pelo modelo vs classe real (10nM, patente)", y=1.01)
plt.tight_layout()
plt.savefig("final_results/fig_confusion_matrices_vs_real.png", bbox_inches="tight")
plt.show()


In [ ]:
# OligoFormer (filtrado) vs DSIR-19nt -- modelo contra modelo, casados pelo "Duplex Name"
merged_ol_f_dsir19 = val_ol_f_10nM["merged"][["Duplex Name", "model_score"]].merge(
    val_dsir19_10nM["merged"][["Duplex Name", "model_score"]],
    on="Duplex Name", suffixes=("_ol_f", "_dsir19"),
)

classe_ol_f = (merged_ol_f_dsir19["model_score_ol_f"] >= LIMIAR_OL).astype(int)
classe_dsir19 = (merged_ol_f_dsir19["model_score_dsir19"] >= LIMIAR_DSIR).astype(int)

fig, ax = plt.subplots(figsize=(5, 4.5))
cm = confusion_matrix(classe_ol_f, classe_dsir19, labels=[1, 0])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Efetivo", "Não efetivo"])
disp.plot(ax=ax, colorbar=False, cmap="Blues", values_format="d")
ax.set_xlabel("DSIR-19nt")
ax.set_ylabel("OligoFormer (filtrado)")
ax.set_title(f"OligoFormer (filtrado) vs DSIR-19nt (n={len(merged_ol_f_dsir19)})", fontsize=10)
plt.tight_layout()
plt.savefig("final_results/fig_confusion_ol_f_vs_dsir19.png", bbox_inches="tight")
plt.show()

classificacao_binaria_10nM["oligoformer_filtrado_vs_dsir_19nt"] = {
    "n": len(merged_ol_f_dsir19),
    "confusion_matrix": cm,
}


### 7.12 F1 score de cada modelo (classe "efetivo" vs eficácia real)

In [ ]:
for nome, merged, limiar in modelos_vs_real:
    classe_real, classe_modelo = classes_binarias(merged, limiar)
    f1 = f1_score(classe_real, classe_modelo)
    classificacao_binaria_10nM[nome]["f1"] = f1
    print(f"{nome}: F1 = {f1:.3f}")


### 7.13 Métricas completas (precisão, recall, F1, acurácia, baseline)

Vale adicionar, sim — o F1 sozinho é enganoso sem esse contexto. O do DSIR (~0.10) parece "o
modelo não presta", mas o recall mostra que é o threshold quase nunca disparando, não ausência de
sinal. E o F1 de um baseline ingênuo ("sempre prever efetivo", ignorando o modelo) mostra que o
OligoFormer filtrado mal supera esse chute (0.563 vs 0.596 do baseline). Sem essa tabela, dá pra
tirar a conclusão errada nos dois casos.

In [ ]:
from sklearn.metrics import precision_score, recall_score

linhas_metricas = []
for nome, merged, limiar in modelos_vs_real:
    classe_real, classe_modelo = classes_binarias(merged, limiar)
    prevalencia = classe_real.mean()
    precisao = precision_score(classe_real, classe_modelo, zero_division=0)
    recall = recall_score(classe_real, classe_modelo, zero_division=0)
    f1 = f1_score(classe_real, classe_modelo, zero_division=0)
    acuracia = (classe_real == classe_modelo).mean()
    f1_baseline = 2 * prevalencia / (prevalencia + 1)  # "sempre prever efetivo": precisão=prevalência, recall=1

    linhas_metricas.append({
        "modelo": nome, "threshold": limiar, "n": len(merged), "prevalencia_real": prevalencia,
        "precisao": precisao, "recall": recall, "f1": f1, "acuracia": acuracia,
        "f1_baseline_sempre_efetivo": f1_baseline,
    })
    classificacao_binaria_10nM[nome].update({
        "prevalencia_real": prevalencia, "precisao": precisao, "recall": recall,
        "acuracia": acuracia, "f1_baseline_sempre_efetivo": f1_baseline,
    })

df_metricas_classificacao = pd.DataFrame(linhas_metricas).set_index("modelo")
display(df_metricas_classificacao.round(3))


### 7.14 Threshold que maximiza o F1 (por modelo)

Varre todos os valores de `model_score` observados como candidatos a corte e escolhe o que dá o
maior F1 — um teto empírico do que cada modelo conseguiria classificar **nesse mesmo conjunto**
usado pra medir o F1. Isso é otimista por construção (o corte é escolhido olhando os mesmos dados
em que é avaliado depois — não é um threshold pra usar "às cegas" em dados novos), então serve como
diagnóstico — "o modelo tem mais sinal do que o threshold da literatura sugere?" — não como
substituto do threshold original.

In [ ]:
def melhor_threshold_f1(merged_df, limiar_original):
    classe_real, classe_modelo_original = classes_binarias(merged_df, limiar_original)
    f1_original = f1_score(classe_real, classe_modelo_original, zero_division=0)

    candidatos = np.unique(merged_df["model_score"])
    melhor_t, melhor_f1 = limiar_original, f1_original
    for t in candidatos:
        classe_pred = (merged_df["model_score"] >= t).astype(int)
        f1 = f1_score(classe_real, classe_pred, zero_division=0)
        if f1 > melhor_f1:
            melhor_t, melhor_f1 = t, f1
    return f1_original, melhor_t, melhor_f1


thresholds_otimizados_10nM = {}
for nome, merged, limiar_original in modelos_vs_real:
    f1_original, melhor_t, melhor_f1 = melhor_threshold_f1(merged, limiar_original)
    thresholds_otimizados_10nM[nome] = {
        "threshold_original": limiar_original, "f1_threshold_original": f1_original,
        "threshold_otimizado": float(melhor_t), "f1_otimizado": melhor_f1,
    }
    print(f"{nome}: original={limiar_original} (F1={f1_original:.3f})  ->  ótimo={melhor_t:.3f} (F1={melhor_f1:.3f})")


### 7.15 DSIR com threshold alternativo (0.8)

0.9 é o default do paper original do DSIR, mas 0.8 também aparece na literatura como corte pra
siRNA efetivo. Reaplica a mesma classificação só pro DSIR com esse threshold, pra ver se o recall
quase-zero da 7.11/7.12 é sensível a essa escolha ou se é estrutural (o score do DSIR raramente
passa nem de 0.8 nesse conjunto).

In [ ]:
import os
os.makedirs("final_results", exist_ok=True)

LIMIAR_DSIR_ALT = 0.8

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
resultados_dsir_alt = {}
for ax, (nome, merged, _) in zip(axes, [m for m in modelos_vs_real if m[0].startswith("DSIR")]):
    classe_real, classe_modelo = classes_binarias(merged, LIMIAR_DSIR_ALT)
    plot_confusion(ax, classe_real, classe_modelo, f"{nome} (threshold=0.8)\n(n={len(merged)})")
    resultados_dsir_alt[nome] = {
        "precisao": precision_score(classe_real, classe_modelo, zero_division=0),
        "recall": recall_score(classe_real, classe_modelo, zero_division=0),
        "f1": f1_score(classe_real, classe_modelo, zero_division=0),
    }

plt.suptitle("DSIR com threshold alternativo (0.8, outro valor citado na literatura)", y=1.02)
plt.tight_layout()
plt.savefig("final_results/fig_confusion_dsir_threshold08.png", bbox_inches="tight")
plt.show()

for nome, m in resultados_dsir_alt.items():
    print(f"{nome}: precisão={m['precisao']:.3f} recall={m['recall']:.3f} F1={m['f1']:.3f}")


### 7.16 Comparação com a literatura (siDPT)

Zhang, Gao & Lai (2025) — *siDPT: siRNA Efficacy Prediction via Debiased Preference-Pair
Transformer* (arXiv:2509.15664) — fazem o mesmo tipo de teste deste notebook: avaliam DSIR e
OligoFormer num dataset "de casa" (onde os modelos performam bem) e depois num **dataset de
patente, fora da distribuição de treino** (o cenário mais parecido com a nossa validação PCSK9).

Números tirados direto do paper (Tabelas 1 e 2), guardados aqui só como referência — não
recalculados, é literatura externa:

- **Datasets públicos "de casa"** (Huesken/Takayuki): correlação de Pearson forte (0.58–0.67) pros
  três métodos, incluindo DSIR e OligoFormer.
- **Dataset de patente, zero-shot** (KHK/CTNNB1/TMPRSS6): a correlação de DSIR e OligoFormer
  despenca pra entre -0.09 e 0.21 — e até o modelo novo do próprio paper (siDPT, desenhado
  especificamente pra esse problema) só chega a 0.19–0.50 nesse cenário.

Isso é o mesmo padrão que a 7.8/7.9 mostraram aqui: concordância forte entre modelos, correlação
bem mais fraca contra dado real e não visto. Serve de calibração pra saber se os nossos números
(seção 7.9) são uma anomalia da nossa análise ou o comportamento esperado do campo — spoiler: é o
segundo.

In [ ]:
# Números da literatura (Zhang et al. 2025, siDPT) -- não calculados aqui, só referência externa.
comparacao_literatura_sidpt = {
    "fonte": "Zhang, Gao & Lai (2025) - siDPT: siRNA Efficacy Prediction via Debiased "
             "Preference-Pair Transformer, arXiv:2509.15664",
    "datasets_publicos_in_distribution": {
        # Tabela 1 do paper -- modelos treinados/testados no mesmo dataset de origem
        "huesken": {
            "dsir": {"auc": 0.8434, "f1": 0.7165, "pearson": 0.6272},
            "oligoformer": {"auc": 0.8725, "f1": 0.8123, "pearson": 0.6688},
            "sidpt": {"auc": 0.8873, "f1": 0.8339, "pearson": 0.6741},
        },
        "takayuki": {
            "dsir": {"auc": 0.7702, "f1": 0.5422, "pearson": 0.5815},
            "oligoformer": {"auc": 0.8628, "f1": 0.5769, "pearson": 0.6596},
            "sidpt": {"auc": 0.8519, "f1": 0.6096, "pearson": 0.6624},
        },
    },
    "datasets_patente_zero_shot": {
        # Tabela 2 do paper -- modelos testados numa patente fora da distribuição de treino
        "khk": {
            "dsir": {"auc": 0.5828, "f1": 0.4706, "pearson": -0.0328},
            "oligoformer": {"auc": 0.700, "f1": 0.5833, "pearson": 0.2081},
            "sidpt": {"auc": 0.8251, "f1": 0.4944, "pearson": 0.4967},
        },
        "ctnnb1": {
            "dsir": {"auc": 0.5770, "f1": 0.6351, "pearson": 0.1891},
            "oligoformer": {"auc": 0.5396, "f1": 0.6809, "pearson": 0.0191},
            "sidpt": {"auc": 0.5948, "f1": 0.7537, "pearson": 0.1946},
        },
        "tmprss6": {
            "dsir": {"auc": 0.5348, "f1": 0.400, "pearson": 0.030},
            "oligoformer": {"auc": 0.5951, "f1": 0.0, "pearson": -0.0936},
            "sidpt": {"auc": 0.7338, "f1": 0.4444, "pearson": 0.4149},
        },
    },
    # Nosso próprio resultado (PCSK9, 10nM), pros mesmos números lado a lado -- puxado das
    # variáveis já calculadas nas seções 6/7, não digitado de novo.
    "nosso_resultado_pcsk9_10nM": {
        "oligoformer_sem_filtro": {
            "spearman_rho": val_ol_10nM["spearman_rho"],
            "f1_threshold_literatura": classificacao_binaria_10nM["OligoFormer (sem filtro)"]["f1"],
            "f1_threshold_otimizado": thresholds_otimizados_10nM["OligoFormer (sem filtro)"]["f1_otimizado"],
        },
        "oligoformer_filtrado": {
            "spearman_rho": val_ol_f_10nM["spearman_rho"],
            "f1_threshold_literatura": classificacao_binaria_10nM["OligoFormer (filtrado)"]["f1"],
            "f1_threshold_otimizado": thresholds_otimizados_10nM["OligoFormer (filtrado)"]["f1_otimizado"],
        },
        "dsir_19nt": {
            "spearman_rho": val_dsir19_10nM["spearman_rho"],
            "f1_threshold_literatura": classificacao_binaria_10nM["DSIR-19nt"]["f1"],
            "f1_threshold_otimizado": thresholds_otimizados_10nM["DSIR-19nt"]["f1_otimizado"],
        },
        "dsir_21nt": {
            "spearman_rho": val_dsir21_10nM["spearman_rho"],
            "f1_threshold_literatura": classificacao_binaria_10nM["DSIR-21nt"]["f1"],
            "f1_threshold_otimizado": thresholds_otimizados_10nM["DSIR-21nt"]["f1_otimizado"],
        },
    },
}

print("Comparação com siDPT (Zhang et al. 2025) montada.")
print("Pearson no cenário zero-shot (patente), por gene, no paper:")
for grupo, alvos in comparacao_literatura_sidpt["datasets_patente_zero_shot"].items():
    print(f"  {grupo}: " + ", ".join(f"{m}={v['pearson']:.3f}" for m, v in alvos.items()))


### 7.17 Salvando os resultados (2 arquivos JSON)

Substitui as duas seções de salvamento antigas que existiam mais abaixo ("## 7. Salvando os
resultados da validação PCSK9" e "## 11. Salvando TODOS os resultados") — ambas quebradas, porque
referenciavam variáveis de uma versão anterior do notebook que não existem mais (`J_pcsk9`,
`val_dsir`, `resumo_19_vs_21`, etc.). Esta seção reconstrói o salvamento do zero, usando só os
dicts/DataFrames que já existem e estão corretos nas seções 4-7 — nada é recalculado aqui, só
reorganizado.

Separado em **2 arquivos**, pra não misturar métricas com dado bruto por siRNA:

- **`resultados_metricas.json`** — só números-resumo (Jaccard/Overlap/Dice, Spearman/Kendall,
  top-N, IC bootstrap, sensibilidade do matching, resumo dos 3 C's) — pequeno, fácil de ler.
- **`resultados_scores.json`** — as tabelas grandes, por siRNA: score de cada fita do MAPT e do
  PCSK9 (DSIR e OligoFormer), os pares score-vs-eficácia-real usados na validação com a patente
  (seção 6), e os pares de rank usados nos scatters DSIR-vs-OligoFormer (seção 7.8).

In [ ]:
def _serialize_json(obj):
    """Serializador genérico para tipos que o json padrão não entende (numpy, pandas, set)."""
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, pd.DataFrame):
        return obj.to_dict(orient="records")
    if isinstance(obj, pd.Series):
        return obj.to_dict()
    if isinstance(obj, set):
        return sorted(obj)
    raise TypeError(f"Tipo não serializável: {type(obj)}")


# Reúne, por dose e por modelo, tudo que a seção 6 já calculou (cobertura, correlação, concordância)
validacao_por_modelo = {
    "10nM": {
        "oligoformer_sem_filtro": dict(n_matched=n_matched_ol, n_confirmado_sequencia=n_confirmed_ol,
                                        jaccard=jaccard_ol_10nM, overlap=overlap_ol_10nM, dice=dice_ol_10nM,
                                        val=val_ol_10nM, topn=top_n_ol_10nM),
        "oligoformer_filtrado": dict(n_matched=n_matched_ol_f, n_confirmado_sequencia=n_confirmed_ol_f,
                                      jaccard=jaccard_ol_f_10nM, overlap=overlap_ol_f_10nM, dice=dice_ol_f_10nM,
                                      val=val_ol_f_10nM, topn=top_n_ol_f_10nM),
        "dsir_19nt": dict(n_matched=n_matched_dsir19, n_confirmado_sequencia=n_confirmed_dsir19,
                           jaccard=jaccard_dsir19_10nM, overlap=overlap_dsir19_10nM, dice=dice_dsir19_10nM,
                           val=val_dsir19_10nM, topn=top_n_dsir19_10nM),
        "dsir_21nt": dict(n_matched=n_matched_dsir21, n_confirmado_sequencia=n_confirmed_dsir21,
                           jaccard=jaccard_dsir21_10nM, overlap=overlap_dsir21_10nM, dice=dice_dsir21_10nM,
                           val=val_dsir21_10nM, topn=top_n_dsir21_10nM),
    },
    "0.1nM": {
        "oligoformer_sem_filtro": dict(n_matched=n_matched_ol_01nM, n_confirmado_sequencia=n_confirmed_ol_01nM,
                                        jaccard=jaccard_ol_01nM, overlap=overlap_ol_01nM, dice=dice_ol_01nM,
                                        val=val_ol_01nM, topn=top_n_ol_01nM),
        "oligoformer_filtrado": dict(n_matched=n_matched_ol_f_01nM, n_confirmado_sequencia=n_confirmed_ol_f_01nM,
                                      jaccard=jaccard_ol_f_01nM, overlap=overlap_ol_f_01nM, dice=dice_ol_f_01nM,
                                      val=val_ol_f_01nM, topn=top_n_ol_f_01nM),
        "dsir_19nt": dict(n_matched=n_matched_dsir19_01nM, n_confirmado_sequencia=n_confirmed_dsir19_01nM,
                           jaccard=jaccard_dsir19_01nM, overlap=overlap_dsir19_01nM, dice=dice_dsir19_01nM,
                           val=val_dsir19_01nM, topn=top_n_dsir19_01nM),
        "dsir_21nt": dict(n_matched=n_matched_dsir21_01nM, n_confirmado_sequencia=n_confirmed_dsir21_01nM,
                           jaccard=jaccard_dsir21_01nM, overlap=overlap_dsir21_01nM, dice=dice_dsir21_01nM,
                           val=val_dsir21_01nM, topn=top_n_dsir21_01nM),
    },
}

df_patent_por_dose = {"10nM": df_patent_10nM, "0.1nM": df_patent_0_1nM}

# Separa, por modelo/dose: métricas (vai pro resultados_metricas.json) e a tabela de pares
# score-vs-eficácia-real por siRNA (vai pro resultados_scores.json)
validacao_patente_metricas = {}
validacao_patente_scores = {}

for dose_label, modelos in validacao_por_modelo.items():
    validacao_patente_metricas[dose_label] = {"n_total_patente": len(df_patent_por_dose[dose_label])}
    validacao_patente_scores[dose_label] = {}
    for modelo_nome, info in modelos.items():
        validacao_patente_metricas[dose_label][modelo_nome] = {
            "cobertura": {
                "n_matched": info["n_matched"],
                "n_confirmado_por_sequencia": info["n_confirmado_sequencia"],
                "jaccard": info["jaccard"],
                "overlap": info["overlap"],
                "dice": info["dice"],
            },
            "correlacao": {k: v for k, v in info["val"].items() if k != "merged"},
            "concordancia_topN": info["topn"],
        }
        validacao_patente_scores[dose_label][modelo_nome] = info["val"]["merged"]

print("Validação organizada por dose/modelo:", list(validacao_por_modelo.keys()), "x", list(validacao_por_modelo["10nM"].keys()))


In [ ]:
import os
os.makedirs("final_results", exist_ok=True)

resultados_metricas = {
    "mapt": {
        "dsir_vs_oligoformer": resultados_finais,
    },
    "pcsk9": {
        "dsir_vs_oligoformer_batch": resultados_concordancia_pcsk9_batch,
        "correlacao_rank_dsir_vs_oligoformer_batch": {
            k: v for k, v in corr_ol_batch_pcsk9.items() if k != "merged"
        },
        "nao_determinismo_oligoformer_batch": {
            "sem_filtro": {
                "n_sirnas": len(df_ol_batch),
                "efficacy_std_describe": df_ol_batch["efficacy_std"].describe(),
            },
            "filtrado": {
                "n_sirnas": len(df_ol_batch_filtered),
                "efficacy_std_describe": df_ol_batch_filtered["efficacy_std"].describe(),
            },
            "comparacao_sem_filtro_vs_filtrado": {
                "n_sirnas_comuns": len(common_sirnas_var),
                "std_medio_sem_filtro": std_unf.mean(),
                "std_medio_filtrado": std_f.mean(),
            },
        },
        "validacao_com_patente": validacao_patente_metricas,
        "resumo_3_cs": {
            "10nM": c3_10nM.to_dict(orient="index"),
            "0.1nM": c3_01nM.to_dict(orient="index"),
        },
        "concordancia_topN_matriz": {
            "10nM": topn_df_10nM.to_dict(orient="index"),
            "0.1nM": topn_df_01nM.to_dict(orient="index"),
        },
        "correlacao_com_ic_bootstrap": df_ci_rho.to_dict(orient="records"),
        "sensibilidade_matching_tolerancia": df_sensibilidade.to_dict(orient="records"),
        "classificacao_binaria_10nM": classificacao_binaria_10nM,
        "thresholds_otimizados_10nM": thresholds_otimizados_10nM,
        "dsir_threshold_alternativo_08": resultados_dsir_alt,
    },
    "comparacao_literatura": comparacao_literatura_sidpt,
}

with open("final_results/resultados_metricas.json", "w", encoding="utf-8") as f:
    json.dump(resultados_metricas, f, indent=2, ensure_ascii=False, default=_serialize_json)

print("Métricas salvas em resultados_metricas.json")


In [ ]:
import os
os.makedirs("final_results", exist_ok=True)

resultados_scores = {
    "mapt": {
        "dsir_scores": df_dsir,
        "oligoformer_scores": df_ol,
        "pares_correlacao_rank_dsir_vs_oligoformer": corr["merged"],
    },
    "pcsk9": {
        "dsir_19nt_scores": df_dsir_pcsk9,
        "dsir_21nt_candidatos_brutos": raw_dsir21_pcsk9,  # sem tabela agregada equivalente (só 1 execução)
        "oligoformer_batch_sem_filtro_scores": df_ol_batch,
        "oligoformer_batch_filtrado_scores": df_ol_batch_filtered,
        "pares_correlacao_rank_dsir_vs_oligoformer_batch": corr_ol_batch_pcsk9["merged"],
        "pares_validacao_com_patente": validacao_patente_scores,
    },
}

with open("final_results/resultados_scores.json", "w", encoding="utf-8") as f:
    json.dump(resultados_scores, f, indent=2, ensure_ascii=False, default=_serialize_json)

print("Scores por siRNA salvos em resultados_scores.json")
